In [ ]:
# # ── 1. Create environment (Python 3.10 matches the repo's .pyc files) ──────
# conda create -n ortrack python=3.10 -y
# conda activate ortrack

# # ── 2. C-level deps before any pip installs ────────────────────────────────
# conda install -c conda-forge libjpeg-turbo -y   # required by jpeg4py

# # ── 3. PyTorch — upgraded from repo's cu102 to cu121 for YOLOv11 compat ────
# pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
#     --index-url https://download.pytorch.org/whl/cu121

# # ── 4. YOLOv11n (ultralytics >= 8.3 ships YOLO11) ─────────────────────────
# pip install ultralytics

# # ── 5. ORTrack requirements (pinned where safe, loosened where needed) ──────
# pip install \
#     easydict==1.10 \
#     gdown==5.2.0 \
#     jpeg4py==0.1.4 \
#     lmdb==1.4.0 \
#     matplotlib==3.7.5 \
#     numpy==1.24.4 \
#     opencv-python==4.7.0.68 \
#     pandas==1.5.2 \
#     Pillow==9.4.0 \
#     pycocotools==2.0.6 \
#     PyYAML==6.0 \
#     scipy \
#     Shapely==2.0.5 \
#     six==1.16.0 \
#     tensorboardX==2.5.1 \
#     timm==0.9.10 \
#     tqdm==4.64.1 \
#     typing_extensions==4.12.2 \
#     wandb==0.13.9

# # ── 6. Setup the package ────────────────────────────────────────────────────
# cd ORTrack
# python tracking/create_default_local_file.py --workspace_dir . --data_dir ./data --save_dir ./output

In [1]:
import sys, os

ORTRACK_ROOT = os.path.abspath("../external/SiamABC")
sys.path.insert(0, ORTRACK_ROOT)
sys.path.append("..")

import torch
import numpy as np
import cv2

import json
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json" , "r") as f:
    manifest = json.load(f)
manifest.keys()

dict_keys(['train', 'public_lb'])

In [3]:
import os
import gc
import cv2
import numpy as np
from tqdm import tqdm
from typing import List

# ─────────────────────────────────────────────────────────────────────────────
# Layout constants
# ─────────────────────────────────────────────────────────────────────────────
PANEL_W = 240
GAP     = 8
FPS_DEFAULT = 30
FONT    = cv2.FONT_HERSHEY_SIMPLEX

C_GT     = (255, 120,   0)
C_SEARCH = (0,   200, 255)
C_UPDATE = (0,   200,  80)
C_FRAME  = (180, 180, 180)
# ── colours — add these two near the other C_ constants ──────────────────
C_YOLO_CANDIDATE  = (0, 200, 255)   # amber  — YOLO box during occlusion
C_YOLO_DISTRACTOR = (0, 0, 180)     # dark red — overlaps held box (distractor)

def _confidence_color(score: float):
    s = max(0.0, min(1.0, score))
    return (0, int(255 * s), int(255 * (1.0 - s)))

def _stamp_panel(panel: np.ndarray, label: str, updated: bool) -> None:
    bar_color = (0, 70, 0) if updated else (30, 30, 30)
    cv2.rectangle(panel, (0, 0), (panel.shape[1], 22), bar_color, -1)
    cv2.putText(panel, label, (5, 15), FONT, 0.38, (255, 255, 255), 1, cv2.LINE_AA)

def _resize_into(src: np.ndarray, dst: np.ndarray) -> None:
    np.copyto(dst, cv2.resize(src, (dst.shape[1], dst.shape[0])))

def _draw_legend(canvas: np.ndarray, x: int, y: int) -> None:
    entries = [
        (C_GT,          "GT bbox (init)"),
        ((0, 255,   0), "Pred  conf=1.0"),
        ((0, 128, 255), "Pred  conf=0.5"),
        ((0,   0, 255), "Pred  conf=0.0"),
        (C_SEARCH,      "Search region"),
        ((0, 0, 255),  "YOLO ROI"),      # ← add this
    ]
    for i, (color, text) in enumerate(entries):
        iy = y + i * 16
        cv2.rectangle(canvas, (x, iy - 9), (x + 12, iy + 3), color, -1)
        cv2.putText(canvas, text, (x + 16, iy), FONT, 0.35, (210, 210, 210), 1, cv2.LINE_AA)

def _refresh_panels(tracker, dyn_image, panel_template, panel_search, frame_idx, updated):
    from utils.bbox_utils import get_extended_crop, extend_bbox
    dyn_bbox = tracker.running_dynamic_bbox
    cfg      = tracker.tracking_config
    ih, iw   = dyn_image.shape[:2]
    pad_val  = np.mean(dyn_image, axis=(0, 1))

    t_ctx = extend_bbox(dyn_bbox, image_width=iw, image_height=ih, offset=cfg["template_bbox_offset"])
    t_crop, _, _ = get_extended_crop(image=dyn_image, bbox=dyn_bbox, context=t_ctx,
                                     crop_size=cfg["template_size"], padding_value=pad_val)
    _resize_into(t_crop, panel_template)
    _stamp_panel(panel_template, f"TEMPLATE  F:{frame_idx}", updated=updated)

    s_ctx = extend_bbox(dyn_bbox, image_width=iw, image_height=ih, offset=cfg["search_context"])
    s_crop, _, _ = get_extended_crop(image=dyn_image, bbox=dyn_bbox, context=s_ctx,
                                     crop_size=cfg["instance_size"], padding_value=pad_val)
    _resize_into(s_crop, panel_search)
    _stamp_panel(panel_search, "SEARCH CTX", updated=updated)



C_OCCLUDED        = (0, 0, 220)     # red border when in DAM occlusion mode
C_YOLO_CANDIDATE  = (0, 200, 255)   # amber  — YOLO box during occlusion
C_YOLO_DISTRACTOR = (0, 0, 180)     # dark red — overlaps held box (distractor)
C_STATUS_OCC      = (0, 0, 200)     # red   filled pill — occlusion
C_STATUS_OK       = (0, 180, 0)     # green filled pill — tracking OK
C_STATUS_TEXT     = (255, 255, 255) # white label inside pill

C_OCCLUDED = (0, 0, 220)   # red border when in DAM occlusion mode  ← defined INSIDE the loop

# but _draw_status_pill uses:
C_STATUS_OCC = (0, 0, 200)  # ← defined OUTSIDE run_inference entirely


def _draw_status_pill(canvas: np.ndarray, in_occlusion: bool, y_offset: int):
    label  = "OCCLUDED" if in_occlusion else "TRACKING"
    color  = C_STATUS_OCC if in_occlusion else C_STATUS_OK
    px, py = 8, y_offset + 8
    pw, ph = 130, 26
    radius = ph // 2
    # filled rectangle body
    cv2.rectangle(canvas, (px + radius, py), (px + pw - radius, py + ph), color, -1)
    # left and right circle caps
    cv2.circle(canvas, (px + radius,      py + radius), radius, color, -1)
    cv2.circle(canvas, (px + pw - radius, py + radius), radius, color, -1)
    # label centred in pill
    (tw, th), _ = cv2.getTextSize(label, FONT, 0.45, 1)
    tx = px + (pw - tw) // 2
    ty = py + (ph + th) // 2 - 1
    cv2.putText(canvas, label, (tx, ty), FONT, 0.45, C_STATUS_TEXT, 1, cv2.LINE_AA)
def run_inference(
    initial_bbox: List[int],
    video_path: str,
    tracker,
    output_path: str = "outputs/tracked_video.mp4",
):
    is_dam        = hasattr(tracker, 'tracker')          # ← correct attribute
    inner_tracker = tracker.tracker if is_dam else tracker

    initial_bbox = np.array(initial_bbox).astype(int)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    head, tail = os.path.split(output_path)
    bbox_dir   = os.path.join(head, "bboxes")
    os.makedirs(bbox_dir, exist_ok=True)
    bbox_file  = os.path.join(bbox_dir, os.path.splitext(tail)[0] + ".txt")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps          = cap.get(cv2.CAP_PROP_FPS) or FPS_DEFAULT
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ret, first_bgr = cap.read()
    if not ret or first_bgr is None:
        cap.release()
        raise RuntimeError(f"Cannot read first frame: {video_path}")

    h, w     = first_bgr.shape[:2]
    canvas_h = max(h, PANEL_W * 2 + GAP)
    total_w  = w + PANEL_W
    y_off    = (canvas_h - h) // 2

    canvas         = np.zeros((canvas_h, total_w, 3), dtype=np.uint8)
    panel_template = np.zeros((PANEL_W, PANEL_W, 3), dtype=np.uint8)
    panel_search   = np.zeros((PANEL_W, PANEL_W, 3), dtype=np.uint8)
    row_tmpl       = np.s_[0             : PANEL_W,            w : total_w]
    row_search     = np.s_[PANEL_W + GAP : PANEL_W * 2 + GAP,  w : total_w]

    avi_path = os.path.splitext(output_path)[0] + "_tmp.avi"
    writer   = cv2.VideoWriter(avi_path, cv2.VideoWriter_fourcc(*"XVID"),
                               fps, (total_w, canvas_h))

    # ── all colours defined locally so nothing is out of scope ───────────────
    C_OCCLUDED        = (0,   0,   220)   # red   — bbox + panel border in occlusion
    C_YOLO_CANDIDATE  = (0,   200, 255)   # amber — YOLO candidate
    C_YOLO_DISTRACTOR = (0,   0,   180)   # dark red — distractor YOLO box
    C_STATUS_OCC      = (0,   0,   200)   # red   pill fill
    C_STATUS_OK       = (0,   180, 0  )   # green pill fill
    C_STATUS_TEXT     = (255, 255, 255)   # white pill text

    # ── pill helper — fully self-contained, no external refs ─────────────────
    def _draw_status_pill(canvas: np.ndarray, in_occlusion: bool) -> None:
        label  = "OCCLUDED" if in_occlusion else "TRACKING"
        color  = C_STATUS_OCC if in_occlusion else C_STATUS_OK
        px     = 8
        py     = y_off + 8          # sits just inside the top-left of the frame
        pw, ph = 140, 28
        r      = ph // 2
        cv2.rectangle(canvas, (px + r, py),      (px + pw - r, py + ph), color, -1)
        cv2.circle  (canvas,  (px + r, py + r),   r, color, -1)
        cv2.circle  (canvas,  (px + pw - r, py + r), r, color, -1)
        (tw, th), _ = cv2.getTextSize(label, FONT, 0.50, 1)
        cv2.putText(canvas, label,
                    (px + (pw - tw) // 2, py + (ph + th) // 2 - 1),
                    FONT, 0.50, C_STATUS_TEXT, 1, cv2.LINE_AA)

    # ── Init frame ────────────────────────────────────────────────────────────
    tracker.initialize(first_bgr, initial_bbox)
    _refresh_panels(inner_tracker, first_bgr, panel_template, panel_search,
                    frame_idx=0, updated=True)

    tracked_bboxes = [initial_bbox]

    canvas.fill(0)
    canvas[y_off : y_off + h, :w] = first_bgr
    canvas[row_tmpl]              = panel_template
    canvas[row_search]            = panel_search
    bx, by, bw, bh = map(int, initial_bbox)
    cv2.rectangle(canvas, (bx, by + y_off), (bx + bw, by + bh + y_off), C_GT, 2)
    _draw_status_pill(canvas, in_occlusion=False)
    cv2.putText(canvas, "F:0  INIT", (156, y_off + 22), FONT, 0.45, C_FRAME, 1, cv2.LINE_AA)
    _draw_legend(canvas, w + 4, canvas_h - 90)
    writer.write(canvas)

    last_dyn_obj = inner_tracker.running_dynamic_image

    # ── Main loop ─────────────────────────────────────────────────────────────
    last_dyn_bbox = inner_tracker.running_dynamic_bbox.copy()
    import time

    times_normal    = []
    times_occlusion = []
    frame_times     = []
    try:
        frame_idx = 1
        # pbar      = tqdm(total=total_frames - 1, desc=os.path.basename(video_path))

        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                break

            # ── update ────────────────────────────────────────────────────────
            t0     = time.perf_counter()
            result       = tracker.update(frame)   # only once
            t1     = time.perf_counter()
            elapsed_ms = (t1 - t0) * 1000.0
            frame_times.append(elapsed_ms)

            if result[2] if is_dam else False:   # in_occlusion
                times_occlusion.append(elapsed_ms)
            else:
                times_normal.append(elapsed_ms)

            bbox         = result[0]
            score        = result[1]
            in_occlusion = result[2] if is_dam else False
            yolo_dets    = result[3] if (is_dam and len(result) > 3) else []
            tracked_bboxes.append(bbox)

            # ── panels ────────────────────────────────────────────────────────
            cur_dyn_obj      = inner_tracker.running_dynamic_image

            cur_dyn_bbox   = inner_tracker.running_dynamic_bbox
            template_updated = not np.array_equal(cur_dyn_bbox, last_dyn_bbox)
            
            if template_updated:
                _refresh_panels(inner_tracker, cur_dyn_obj, panel_template, panel_search,
                                frame_idx=frame_idx, updated=True)
                last_dyn_obj  = cur_dyn_obj
                last_dyn_bbox = cur_dyn_bbox.copy()   # ← ADD THIS   
            else:
                _stamp_panel(panel_template, f"Dynamic TEMPLATE  F:{frame_idx - 1}", updated=False)
                _stamp_panel(panel_search,   "SEARCH CTX", updated=False)

            # ── composite (must happen before any drawing) ────────────────────
            canvas.fill(0)
            canvas[y_off : y_off + h, :w] = frame
            canvas[row_tmpl]              = panel_template
            canvas[row_search]            = panel_search

            # ── panel border ──────────────────────────────────────────────────
            if in_occlusion:
                cv2.rectangle(canvas, (w, 0), (total_w-1, canvas_h-1), C_OCCLUDED, 3)
            elif template_updated:
                cv2.rectangle(canvas, (w, 0), (total_w-1, canvas_h-1), C_UPDATE, 2)

            # ── search region mapping ─────────────────────────────────────────
            mapping = inner_tracker.tracking_state.mapping
            if mapping is not None:
                mx, my, mw, mh = map(int, mapping)
                cv2.rectangle(canvas, (mx, my + y_off), (mx+mw, my+mh+y_off), C_SEARCH, 1)

            # ── YOLO candidates ───────────────────────────────────────────────
            if in_occlusion and yolo_dets:
                held = tracker.held_box if is_dam else None
                for det in yolo_dets:
                    dx, dy, dw, dh = map(int, det)
                    is_dist = held is not None and _iou(det, held) >= tracker.tau_occ
                    color   = C_YOLO_DISTRACTOR if is_dist else C_YOLO_CANDIDATE
                    cv2.rectangle(canvas, (dx, dy+y_off), (dx+dw, dy+dh+y_off), color, 1)
                    cv2.putText(canvas, "D" if is_dist else "Y",
                                (dx+2, dy+y_off+12), FONT, 0.38, color, 1, cv2.LINE_AA)

            # ── main bbox ─────────────────────────────────────────────────────
            bx, by, bw, bh = map(int, bbox)
            pred_color = C_OCCLUDED if in_occlusion else _confidence_color(score)
            cv2.rectangle(canvas, (bx, by+y_off), (bx+bw, by+bh+y_off), pred_color, 2)
            cv2.putText(canvas, f"{score:.2f}", (bx, max(by+y_off-4, 12)),
                        FONT, 0.45, pred_color, 1, cv2.LINE_AA)
            # ── status pill — drawn after bbox so it's always on top ──────────
            _draw_status_pill(canvas, in_occlusion=in_occlusion)

            # ── HUD text — x=156 clears the 140px pill ────────────────────────
            if in_occlusion:
                hud = f"F:{frame_idx}  [DAM RECOVERY]"
            elif template_updated:
                hud = f"F:{frame_idx}  [TMPL UPDATE]"
            else:
                hud = f"F:{frame_idx}"
            cv2.putText(canvas, hud, (156, y_off + 22), FONT, 0.45, C_FRAME, 1, cv2.LINE_AA)

            _draw_legend(canvas, w + 4, canvas_h - 90)

            if is_dam and in_occlusion:
                rx, ry, rw, rh = tracker._get_yolo_search_roi(frame)
                cv2.rectangle(canvas,
                            (rx,      ry      + y_off),
                            (rx + rw, ry + rh + y_off),
                            (0, 165, 255),   # orange
                            1)
                cv2.putText(canvas, "YOLO ROI",
                            (rx + 2, ry + y_off + 12),
                            FONT, 0.38, (0, 165, 255), 1, cv2.LINE_AA)

            writer.write(canvas)

        # if in_occlusion:
        #     hud = f"F:{frame_idx}  [DAM RECOVERY]  {elapsed_ms:.1f}ms"
        # elif template_updated:
        #     hud = f"F:{frame_idx}  [TMPL UPDATE]  {elapsed_ms:.1f}ms"
        # else:
        #     hud = f"F:{frame_idx}  {elapsed_ms:.1f}ms"

            frame_idx += 1
            # pbar.update(1)

        # pbar.close()

    finally:
        cap.release()
        writer.release()

    if avi_path != output_path:
        os.replace(avi_path, output_path)

    with open(bbox_file, "w", encoding="utf-8") as f:
        for bb in tracked_bboxes:
            f.write(f"{bb[0]} {bb[1]} {bb[2]} {bb[3]} \n")

    del canvas, panel_template, panel_search


    def _stats(name, arr):
        if not arr:
            print(f"{name:20s}  no data")
            return
        a = np.array(arr)
        print(f"{name:20s}  n={len(a):4d}  "
            f"mean={a.mean():.1f}ms  "
            f"med={np.median(a):.1f}ms  "
            f"p95={np.percentile(a,95):.1f}ms  "
            f"p99={np.percentile(a,99):.1f}ms  "
            f"min={a.min():.1f}ms  "
            f"max={a.max():.1f}ms  "
          f"fps={1000/a.mean():.1f}")

        print("\n─── Latency Report ───────────────────────────────────────")
    _stats("ALL FRAMES",    frame_times)
    _stats("NORMAL TRACK",  times_normal)
    _stats("OCCLUSION",     times_occlusion)
    if times_occlusion:
        print(f"  occlusion frames: {len(times_occlusion)} "
                f"({100*len(times_occlusion)/len(frame_times):.1f}% of total)")
        
    print("──────────────────────────────────────────────────────────\n")
    gc.collect()

In [5]:
import cv2
import numpy as np
from collections import deque
from typing import List, Optional, Tuple
from ultralytics import YOLO


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _extract_descriptor(frame: np.ndarray, bbox, size=16) -> Optional[np.ndarray]:
    """phi(I_t, b) = normalised concat of grayscale patch + HSV histogram."""
    x, y, w, h = map(int, bbox)
    x, y = max(0, x), max(0, y)
    w, h = max(1, w), max(1, h)
    patch = frame[y:y+h, x:x+w]
    if patch.size == 0:
        return None
    gray   = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    p      = cv2.resize(gray, (size, size)).flatten().astype(np.float32)
    hsv    = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 4, 4],
                          [0, 180, 0, 256, 0, 256]).flatten().astype(np.float32)
    desc = np.concatenate([p, h_hist])
    norm = np.linalg.norm(desc)
    return desc / (norm + 1e-8)


def _iou(a, b) -> float:
    ax2, ay2 = a[0] + a[2], a[1] + a[3]
    bx2, by2 = b[0] + b[2], b[1] + b[3]
    ix1 = max(a[0], b[0]);  iy1 = max(a[1], b[1])
    ix2 = min(ax2,  bx2);   iy2 = min(ay2,  by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / (union + 1e-8)


def _cos_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))




class BBoxEKF:
    """
    Center-state EKF.  Internal state is [cx, cy, vx, vy].
    Corners are only used at input (bbox → center) and output (center → bbox).
    This makes velocity completely immune to bbox shrinkage at occlusion entry.
    """

    DIM_X = 4
    DIM_Z = 2   # we observe (cx, cy) only

    def __init__(self, bbox, process_noise: float = 2.0, meas_noise: float = 5.0,
                 vel_window: int = 100):
        x1, y1, w, h = bbox
        cx = float(x1 + w / 2.0)
        cy = float(y1 + h / 2.0)
        self._bw = float(w)
        self._bh = float(h)

        self.x = np.array([cx, cy, 0., 0.])

        self.P = np.diag([10., 10., 100., 100.]).astype(float)

        self.Q = np.diag([
            process_noise,
            process_noise,
            process_noise * 4,
            process_noise * 4,
        ]).astype(float)

        self.R = np.eye(self.DIM_Z, dtype=float) * meas_noise

        # H_mat: extract [cx, cy] from [cx, cy, vx, vy]
        self._H_mat = np.array([
            [1., 0., 0., 0.],
            [0., 1., 0., 0.],
        ], dtype=float)

    # ── Transition Jacobian ───────────────────────────────────────────────────

    def _compute_jacobian(
        self,
        H:          Optional[np.ndarray],
        H_reliable: bool,
        cx:         float,
        cy:         float,
        new_cx_h:   float,
        new_cy_h:   float,
    ) -> np.ndarray:
        """
        4×4 Jacobian of the transition function w.r.t. [cx, cy, vx, vy].

        Without homography:
            cx_new = cx + vx   →   F = [[1,0,1,0],
            cy_new = cy + vy            [0,1,0,1],
                                        [0,0,1,0],
                                        [0,0,0,1]]

        With homography:
            cx_new = f(cx,cy) + vx
            ∂cx_new/∂cx = A  (quotient-rule derivative, same as before)
            ∂cx_new/∂cy = B
            etc.
        """
        F = np.eye(self.DIM_X, dtype=float)
        F[0, 2] = 1.0   # cx_new = cx + vx
        F[1, 3] = 1.0   # cy_new = cy + vy

        if H is not None and H_reliable:
            denom = H[2, 0] * cx + H[2, 1] * cy + H[2, 2] + 1e-8
            A = (H[0, 0] - new_cx_h * H[2, 0]) / denom
            B = (H[0, 1] - new_cx_h * H[2, 1]) / denom
            C = (H[1, 0] - new_cy_h * H[2, 0]) / denom
            D = (H[1, 1] - new_cy_h * H[2, 1]) / denom

            # ∂new_cx_h/∂cx = A,  ∂new_cx_h/∂cy = B  (no 0.5 factor — center is 1:1)
            # ∂new_cy_h/∂cx = C,  ∂new_cy_h/∂cy = D
            F[0, 0] = A;   F[0, 1] = B
            F[1, 0] = C;   F[1, 1] = D

        return F

    # ── Predict ───────────────────────────────────────────────────────────────

    def predict(self, H: Optional[np.ndarray] = None, H_reliable: bool = False):
        cx, cy = self.x[0], self.x[1]

        if H is not None and H_reliable:
            denom    = H[2, 0] * cx + H[2, 1] * cy + H[2, 2]
            new_cx_h = (H[0, 0] * cx + H[0, 1] * cy + H[0, 2]) / (denom + 1e-8)
            new_cy_h = (H[1, 0] * cx + H[1, 1] * cy + H[1, 2]) / (denom + 1e-8)
        else:
            new_cx_h = cx
            new_cy_h = cy

        self.x[0] = new_cx_h + self.x[2]   # cx + vx
        self.x[1] = new_cy_h + self.x[3]   # cy + vy
        # vx, vy unchanged (constant velocity model)

        F      = self._compute_jacobian(H, H_reliable, cx, cy, new_cx_h, new_cy_h)
        self.P = F @ self.P @ F.T + self.Q

    # ── Update ────────────────────────────────────────────────────────────────

    def update(self, bbox):
        x1, y1, w, h = bbox
        cx = float(x1 + w / 2.0)
        cy = float(y1 + h / 2.0)

        # Update running size estimate (EMA) — only size, not position
        self._bw = 0.85 * self._bw + 0.15 * float(w)
        self._bh = 0.85 * self._bh + 0.15 * float(h)

        z     = np.array([cx, cy])
        innov = z - self._H_mat @ self.x
        S     = self._H_mat @ self.P @ self._H_mat.T + self.R
        K     = self.P @ self._H_mat.T @ np.linalg.inv(S)
        self.x     = self.x + K @ innov
        I_KH       = np.eye(self.DIM_X) - K @ self._H_mat
        self.P     = I_KH @ self.P

    # ── Output helpers ────────────────────────────────────────────────────────

    def get_bbox(self) -> np.ndarray:
        cx, cy = self.x[0], self.x[1]
        x1 = cx - self._bw / 2.0
        y1 = cy - self._bh / 2.0
        return np.array([int(x1), int(y1),
                         max(1, int(self._bw)),
                         max(1, int(self._bh))], dtype=int)

    def get_velocity(self) -> np.ndarray:
        return self.x[2:4].copy()

    def get_uncertainty(self) -> float:
        # uncertainty over cx, cy only
        return float(np.sqrt(np.diag(self.P[:2]).mean()))

    def reseed(self, bbox, velocity: np.ndarray):
        x1, y1, w, h = bbox
        self.x[0] = float(x1 + w / 2.0)
        self.x[1] = float(y1 + h / 2.0)
        self.x[2] = float(velocity[0])
        self.x[3] = float(velocity[1])
        self._bw  = float(w)
        self._bh  = float(h)
        self.P    = np.diag([25., 25., 40., 40.]).astype(float)

    def nudge_position(self, bbox):
        x1, y1, w, h = bbox
        self.x[0] = float(x1 + w / 2.0)
        self.x[1] = float(y1 + h / 2.0)
        self._bw  = float(w)
        self._bh  = float(h)


# ─────────────────────────────────────────────────────────────────────────────
# AppearanceMemory  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

class AppearanceMemory:
    """
    RAM  — ring-buffer of geometrically verified target states.
    DRM  — stable anchor buffer promoted from RAM when the last W RAM
           descriptors mutually agree in appearance.
    """

    def __init__(
        self,
        capacity:     int   = 20,
        tau_iou:      float = 0.40,
        tau_area:     float = 0.25,
        drm_capacity: int   = 10,
        tau_sim:      float = 0.85,
        window_W:     int   = 5,
        mmin:         int   = 3,
    ):
        self.capacity = capacity
        self.tau_iou  = tau_iou
        self.tau_area = tau_area
        self._buf: deque = deque(maxlen=capacity)

        self.drm_capacity = drm_capacity
        self.tau_sim      = tau_sim
        self.window_W     = window_W
        self.mmin         = mmin
        self._drm: deque  = deque(maxlen=drm_capacity)
        self._t:   int    = 0

    def reset(self):
        self._buf.clear()
        self._drm.clear()
        self._t = 0

    def try_admit(self, bbox, desc: np.ndarray, prev_bbox) -> bool:
        iou_ok = _iou(bbox, prev_bbox) >= self.tau_iou
        if self._buf:
            med     = float(np.median([e[0][2] * e[0][3] for e in self._buf]))
            area_ok = abs(bbox[2] * bbox[3] - med) / (med + 1e-6) <= self.tau_area
        else:
            area_ok = True
        if iou_ok and area_ok:
            self._buf.append((np.array(bbox, dtype=int), desc.copy()))
            self._t += 1
            self._try_promote_to_drm(bbox, desc)
            return True
        return False

    def _try_promote_to_drm(self, bbox, new_desc: np.ndarray) -> None:
        recent = list(self._buf)[-self.window_W:]
        if len(recent) < self.mmin:
            return
        agreements = sum(
            1 for (_, d) in recent
            if _cos_sim(new_desc, d) >= self.tau_sim
        )
        if agreements >= self.mmin:
            self._drm.append((
                np.array(bbox, dtype=int),
                new_desc.copy(),
                self._t,
            ))

    def best_descriptor(self) -> Optional[np.ndarray]:
        return self._buf[-1][1] if self._buf else None

    def match(self, frame: np.ndarray, candidates: List[np.ndarray],
              threshold: float) -> Tuple[Optional[np.ndarray], float]:
        ref = self.best_descriptor()
        if ref is None or not candidates:
            return None, -1.0
        best_box, best_score = None, -1.0
        for bbox in candidates:
            desc = _extract_descriptor(frame, bbox)
            if desc is None:
                continue
            s = _cos_sim(ref, desc)
            if s > best_score:
                best_score, best_box = s, bbox
        if best_score >= threshold:
            return np.array(best_box, dtype=int), best_score
        return None, best_score

    def drm_match(
        self,
        frame:            np.ndarray,
        candidates:       List[np.ndarray],
        ref_bbox:         np.ndarray,
        velocity:         np.ndarray,
        distractor_bank,
        lam_iou:          float = 0.40,
        lam_app:          float = 0.30,
        lam_mot:          float = 0.20,
        lam_time:         float = 0.10,
        alpha:            float = 0.05,
        gamma:            float = 0.25,
        margin:           float = 0.35,
        top_k:            int   = 3,
        skip_threshold:   float = 0.80,
        search_cx:        Optional[float] = None,
        search_cy:        Optional[float] = None,
        dist_sigma:       Optional[float] = None,
        lam_dist:         float           = 0.15,
        lam_cand_dir:     float           = 0.15,
    ) -> List[Tuple[np.ndarray, float]]:
        if not self._drm:
            box, score = self.match(frame, candidates, threshold=margin)
            if box is not None:
                return [(box, score)]
            return []

        if not candidates:
            return []

        ref_cx   = ref_bbox[0] + ref_bbox[2] / 2.0
        ref_cy   = ref_bbox[1] + ref_bbox[3] / 2.0
        vel_norm = float(np.linalg.norm(velocity)) + 1e-8

        scored: List[Tuple[np.ndarray, float]] = []

        for cand_bbox in candidates:
            cand_desc = _extract_descriptor(frame, cand_bbox)
            if cand_desc is None:
                continue

            anchor_scores = []
            for (dk_bbox, dk_desc, rho_k) in self._drm:
                s_iou = lam_iou * _iou(dk_bbox, cand_bbox)
                s_app = lam_app * _cos_sim(dk_desc, cand_desc)

                dk_cx      = dk_bbox[0] + dk_bbox[2] / 2.0
                dk_cy      = dk_bbox[1] + dk_bbox[3] / 2.0
                motion_vec = np.array([dk_cx - ref_cx, dk_cy - ref_cy])
                mot_norm   = float(np.linalg.norm(motion_vec)) + 1e-8
                pi_t       = float(np.dot(velocity, motion_vec) /
                                   (vel_norm * mot_norm))
                pi_t       = max(0.0, pi_t)
                s_mot      = lam_mot * pi_t

                age    = max(0, self._t - rho_k)
                s_time = lam_time * float(np.exp(-alpha * age))

                raw = s_iou + s_app + s_mot + s_time

                if distractor_bank:
                    pen = max(_cos_sim(dk_desc, nu) for nu in distractor_bank)
                    raw -= gamma * pen

                anchor_scores.append(raw)

            cand_score = max(anchor_scores) if anchor_scores else -np.inf

            if (search_cx is not None and search_cy is not None
                    and dist_sigma is not None and dist_sigma > 0):
                cand_cx    = cand_bbox[0] + cand_bbox[2] / 2.0
                cand_cy    = cand_bbox[1] + cand_bbox[3] / 2.0
                d          = np.hypot(cand_cx - search_cx, cand_cy - search_cy)
                cand_score -= lam_dist * (1.0 - np.exp(-0.5 * (d / dist_sigma) ** 2))


            if lam_cand_dir > 0 and vel_norm > 1e-3:
                cand_cx  = cand_bbox[0] + cand_bbox[2] / 2.0
                cand_cy  = cand_bbox[1] + cand_bbox[3] / 2.0
                cand_vec = np.array([cand_cx - ref_cx, cand_cy - ref_cy])
                cand_d   = float(np.linalg.norm(cand_vec)) + 1e-8
                cos_dir  = float(np.dot(velocity, cand_vec) / (vel_norm * cand_d))
                cand_score += lam_cand_dir * ((cos_dir + 1.0) / 2.0)


            if cand_score > margin:
                scored.append((np.array(cand_bbox, dtype=int), float(cand_score)))

        if not scored:
            return []

        scored.sort(key=lambda x: x[1], reverse=True)

        if scored[0][1] >= skip_threshold:
            return [scored[0]]

        return scored[:top_k]

    def drm_size(self) -> int:
        return len(self._drm)

    def __len__(self):
        return len(self._buf)


# ─────────────────────────────────────────────────────────────────────────────
# EdgeDAMTracker
# ─────────────────────────────────────────────────────────────────────────────

class EdgeDAMTracker:
    """
    Occlusion-aware wrapper around SiamABCTracker with EKF motion estimation,
    background homography compensation, and DRM appearance memory.

    Key design decisions vs previous version
    ─────────────────────────────────────────
    1. MOTION MODEL DISABLED DURING NORMAL TRACKING
       The EKF runs silently (predict + update every frame) to keep its state
       accurate, but its bbox output is NOT used — SiamABC output is returned
       directly.  Velocity is computed from finite differences of stored
       conf_history positions (EMA-smoothed, lookback=3 frames).
       Benefit: no EKF smoothing lag or slight positional bias on the returned
       bbox during normal tracking; EKF state is still fresh and accurate at
       occlusion entry so _rebuild_ekf_from_clean_history works well.

    2. 3-PHASE DISTRIBUTED OCCLUSION LOOP
       Occlusion work is spread across frames instead of done all in one:
         Phase 0 (frame N)  : Steer SiamABC to EKF-predicted position.
                              If score ≥ reacq_threshold → exit occlusion.
         Phase 1 (frame N+1): YOLO detect in uncertainty-driven ROI + DRM
                              match. Store ranked candidates.
         Phase 2 (frame N+2): Verify top DRM candidates with SiamABC
                              (run_track_for_candidate).  Candidates are
                              velocity-compensated by 1 frame before verifying
                              because they were detected on the previous frame.
       If phase 1 produces no candidates, phase 2 is skipped and we return
       straight to phase 0 the next frame.

    3. DYNAMIC SHRINKAGE-ONSET SKIP
       _detect_shrinkage_onset() is called at every occlusion entry (unless
       loss_cause == 'camera_motion') and returns the number of trailing
       conf_history frames corrupted by pre-occlusion bbox shrinkage.  Those
       frames are excluded when rebuilding the EKF and when initialising the
       search centre.

    ─────────────────────────────────────────────────────────────────────────
    RECOMMENDED PARAMETER SETS
    ─────────────────────────────────────────────────────────────────────────

    ── Fast-paced close-range (large objects, high pixel velocity, short clips)
       conf_threshold              = 0.55
       reacq_threshold             = 0.75
       ekf_process_noise           = 12.0   # large → tracks fast velocity changes
       ekf_meas_noise              = 2.0    # trust detections, less smoothing
       roi_start_expand            = 4.0    # start with a wider ROI from frame 1
       yolo_search_expand          = 20.0
       search_expand_growth_factor = 1.5
       search_expand_growth_every  = 3
       app_match_threshold         = 0.0    # appearance can change rapidly at range
       drm_margin                  = 0.10
       drm_top_k                   = 5
       drm_skip_threshold          = 10
       shrinkage_min_drop_frac     = 0.10   # large bbox naturally varies ≥10%
       shrinkage_max_lookback      = 20
       long_distance_area_fraction = 0.005  # object must be bigger to be 'far'
       long_distance_conf_threshold= 0.35
       velocity_lookback           = 2      # shorter lookback for fast motion

    ── General purpose
       conf_threshold              = 0.50
       reacq_threshold             = 0.82
       ekf_process_noise           = 5.0
       ekf_meas_noise              = 4.0
       roi_start_expand            = 2.5
       yolo_search_expand          = 12.0
       search_expand_growth_factor = 1.3
       search_expand_growth_every  = 6
       app_match_threshold         = 0.0
       drm_margin                  = 0.15
       drm_top_k                   = 5
       drm_skip_threshold          = 15
       shrinkage_min_drop_frac     = 0.06
       shrinkage_max_lookback      = 60
       long_distance_area_fraction = 0.004
       long_distance_conf_threshold= 0.35
       velocity_lookback           = 3
    ─────────────────────────────────────────────────────────────────────────
    """

    def __init__(
        self,
        siam_tracker,
        yolo_weights:               str   = "yolo11n.pt",
        conf_threshold:             float = 0.60,
        reacq_threshold:            float = 0.55,
        yolo_conf:                  float = 0.30,
        yolo_iou:                   float = 0.45,
        app_match_threshold:        float = 0.72,
        nudge_alpha:                float = 0.30,
        tau_occ:                    float = 0.40,
        beta:                       float = 0.06,
        mem_capacity:               int   = 20,
        tau_iou:                    float = 0.40,
        tau_area:                   float = 0.25,
        ncc_threshold:              float = 0.70,
        ncc_expand:                 float = 2.5,
        conf_history_len:           int   = 200,
        history_decay:              float = 0.5,
        history_skip_last:          int   = 2,
        # ── ROI params ────────────────────────────────────────────────────
        yolo_search_expand:         float = 5.0,
        roi_start_expand:           float = 1.5,
        size_history_len:           int   = 40,
        # ── DRM params ────────────────────────────────────────────────────
        drm_capacity:               int   = 8,
        drm_tau_sim:                float = 0.85,
        drm_window_W:               int   = 10,
        drm_mmin:                   int   = 3,
        drm_lam_iou:                float = 0.40,
        drm_lam_app:                float = 0.30,
        drm_lam_mot:                float = 0.20,
        drm_lam_time:               float = 0.10,
        drm_alpha:                  float = 0.05,
        drm_gamma:                  float = 0.30,
        drm_margin:                 float = 0.35,
        drm_top_k:                  int   = 3,
        drm_skip_threshold:         float = 0.80,
        drm_lam_dist:               float = 0.15,
        drm_dist_sigma_factor:      float = 2.5,
        # ── EKF params ────────────────────────────────────────────────────
        ekf_process_noise:          float = 2.0,
        ekf_meas_noise:             float = 5.0,
        homo_max_corners:           int   = 200,
        homo_inlier_threshold:      float = 0.50,
        # ── Shrinkage detection params ────────────────────────────────────
        shrinkage_min_drop_frac:    float = 0.06,   # tune per scenario (see docstring)
        shrinkage_max_lookback:     int   = 60,
        # ── Velocity estimation (normal tracking) ─────────────────────────
        velocity_lookback:          int   = 3,       # frames used for finite-diff velocity
        velocity_smooth_alpha:      float = 0.4,     # EMA weight on raw measurement
        # ── Memory leak fix ───────────────────────────────────────────────
        distractor_bank_maxlen:     int   = 50,
        # ── Misc ──────────────────────────────────────────────────────────
        velocity_decay:             float = 0.95,
        search_expand_growth_factor: float = 1.2,
        search_expand_growth_every:  int   = 5,
        search_expand_max:          float = 15.0,
        long_distance_conf_threshold: float = 0.35,
        long_distance_area_fraction:  float = 0.004,
        long_distance_mode:           bool  = False,
        enter_occlusion_on_loss:         bool  = True,
        velocity_window_average = 80,
        occlusion_patience:    int   = 5,    # score must be below threshold for N frames in a row
        occlusion_hysteresis:  float = 0.10,
        drm_lam_cand_dir:           float = 0.15,
    ):
        self.tracker              = siam_tracker
        self.yolo                 = YOLO(yolo_weights)
        self.conf_threshold       = conf_threshold
        self.reacq_threshold      = reacq_threshold
        self.yolo_conf            = yolo_conf
        self.yolo_iou_thr         = yolo_iou
        self.app_match_threshold  = app_match_threshold
        self.nudge_alpha          = nudge_alpha
        self.tau_occ              = tau_occ
        self.beta                 = beta
        self.ncc_threshold        = ncc_threshold
        self.ncc_expand           = ncc_expand
        self.conf_history_len     = conf_history_len
        self.history_decay        = history_decay
        self.history_skip_last    = history_skip_last
        self.yolo_search_expand   = yolo_search_expand
        self.roi_start_expand     = roi_start_expand
        self.size_history_len     = size_history_len

        # EKF
        self.ekf_process_noise     = ekf_process_noise
        self.ekf_meas_noise        = ekf_meas_noise
        self.homo_max_corners      = homo_max_corners
        self.homo_inlier_threshold = homo_inlier_threshold

        # Shrinkage detection
        self.shrinkage_min_drop_frac = shrinkage_min_drop_frac
        self.shrinkage_max_lookback  = shrinkage_max_lookback

        # Velocity (finite-diff, used during normal tracking)
        self.velocity_lookback      = velocity_lookback
        self.velocity_smooth_alpha  = velocity_smooth_alpha

        # Memory leak fix
        self._distractor_bank_maxlen = distractor_bank_maxlen
        self.velocity_window_average = velocity_window_average

        self.search_expand_growth_factor = search_expand_growth_factor
        self.search_expand_growth_every  = search_expand_growth_every
        self.search_expand_max           = search_expand_max
        self._occ_frames: int            = 0

        self._drm_dist_sigma_factor = drm_dist_sigma_factor
        self._size_history: deque   = deque(maxlen=size_history_len)
        self._cam_disp_history: deque = deque(maxlen=conf_history_len)
        self._vel_history: deque = deque(maxlen=200)

        self.long_distance_conf_threshold  = long_distance_conf_threshold
        self.long_distance_area_fraction   = long_distance_area_fraction
        self.long_distance_mode            = long_distance_mode
        self.recovered_early_occlusion= True
        self.enter_occlusion_on_loss = enter_occlusion_on_loss
        self._drm_kwargs = dict(
            lam_iou        = drm_lam_iou,
            lam_app        = drm_lam_app,
            lam_mot        = drm_lam_mot,
            lam_time       = drm_lam_time,
            alpha          = drm_alpha,
            gamma          = drm_gamma,
            margin         = drm_margin,
            top_k          = drm_top_k,
            skip_threshold = drm_skip_threshold,
            lam_dist       = drm_lam_dist,
            lam_cand_dir   = drm_lam_cand_dir,
        )

        self.memory = AppearanceMemory(
            capacity     = mem_capacity,
            tau_iou      = tau_iou,
            tau_area     = tau_area,
            drm_capacity = drm_capacity,
            tau_sim      = drm_tau_sim,
            window_W     = drm_window_W,
            mmin         = drm_mmin,
        )

        # Runtime state
        self.current_bbox:       Optional[np.ndarray] = None
        self.held_box:           Optional[np.ndarray] = None
        self.in_occlusion:       bool                 = False
        self.frame_idx:          int                  = 0
        self.velocity:           np.ndarray           = np.zeros(2)
        self.prev_gray:          Optional[np.ndarray] = None
        self.init_frame:         Optional[np.ndarray] = None
        self.init_bbox:          Optional[np.ndarray] = None
        self._last_yolo:         List                 = []
        self._yolo_cache:        List                 = []
        self._distractor_bank:   deque                = deque(maxlen=distractor_bank_maxlen)
        self._out_of_frame:      bool                 = False
        self._exit_edge:         Optional[str]        = None
        self._search_cx:         Optional[float]      = None
        self._search_cy:         Optional[float]      = None
        self._conf_history:      deque                = deque(maxlen=conf_history_len)
        self._center_history: deque = deque(maxlen=200)
        self._cam_vel_history: deque = deque(maxlen=200)
        # ── 3-phase occlusion state ───────────────────────────────────────
        # Phase 0: SiamABC attempt
        # Phase 1: YOLO + DRM match
        # Phase 2: Tracker verification of candidates from phase 1
        self._occ_phase:          int  = 0
        self._pending_candidates: List = []  # DRM candidates waiting for verification

        self.ekf: Optional[BBoxEKF] = None
        self._last_H:          Optional[np.ndarray] = None
        self._last_H_reliable: bool                 = False

        self._flow_scale   = 0.5
        self._cached_pts   = None
        self._cached_shape = None


        self._low_score_streak = 0
        self._occlusion_patience   = occlusion_patience
        self._occlusion_hysteresis = occlusion_hysteresis
        self._gated_score = 1.0   # what we actually report to EdgeDAM

    # ── Public API ────────────────────────────────────────────────────────────

    def initialize(self, frame: np.ndarray, bbox) -> None:
        self.tracker.enable_tta()
        bbox = np.array(bbox, dtype=int)
        self.tracker.initialize(frame, bbox)
        self.current_bbox     = bbox.copy()
        self.held_box         = bbox.copy()
        self.in_occlusion     = False
        self.frame_idx        = 0
        self.velocity         = np.zeros(2)
        self.prev_gray        = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        self.init_frame       = frame.copy()
        self.init_bbox        = bbox.copy()
        self._distractor_bank = deque(maxlen=self._distractor_bank_maxlen)
        self._out_of_frame    = False
        self._exit_edge       = None
        self._search_cx       = None
        self._search_cy       = None
        self._size_history.clear()
        self.memory.reset()
        self._conf_history.clear()
        self._cam_disp_history.clear()
        self._yolo_cache      = []
        self._occ_frames      = 0
        self._occ_phase       = 0
        self._pending_candidates = []
        self.recovered_early_occlusion=True
        self._last_H          = None        
        self._last_H_reliable = False   
        self._last_yolo  = []     


        self.ekf = BBoxEKF(bbox,
                           process_noise=self.ekf_process_noise,
                           meas_noise=self.ekf_meas_noise)

        desc = _extract_descriptor(frame, bbox)

        self._vel_history.clear()
        self._center_history.clear()
        self._cam_vel_history.clear()
        if desc is not None:
            self.memory.try_admit(bbox, desc, bbox)

    def update(self, frame: np.ndarray) -> Tuple[np.ndarray, float, bool, List]:
        self.frame_idx  += 1
        self._last_yolo  = []
        self._yolo_cache = []

        H, H_reliable, current_gray = self._estimate_homography(frame)
        self._last_H          = H
        self._last_H_reliable = H_reliable

        if self.in_occlusion and self._out_of_frame:
            # Object is outside frame — don't move the EKF position at all.
            # Velocity extrapolation would push it further out every frame,
            # so when the camera returns the EKF is far from the exit edge.
            # Instead just grow uncertainty (P += Q) so confidence degrades
            # naturally, but position stays pinned at the exit boundary.
            self.ekf.P = self.ekf.P + self.ekf.Q
        else:
            self.ekf.predict(H=H, H_reliable=H_reliable)

        if self.in_occlusion:
            bbox, score = self._occlusion_update(frame)
        else:
            bbox, score = self._normal_update(frame)

        self.prev_gray = current_gray

        if self.in_occlusion:
            return np.zeros(4, dtype=int), 0.0, True, self._last_yolo

        return bbox.copy(), float(score), self.in_occlusion, self._last_yolo
    # ── Normal path ───────────────────────────────────────────────────────────

    def _normal_update(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        pred_bbox, score, _ = self.tracker.update(frame)
        pred_bbox = np.array(pred_bbox, dtype=int)

        # ── threshold (long-distance mode) ───────────────────────────────
        if self.frame_idx > 0:
            long_distanced_object = self._is_long_distance(frame)
            effective_threshold = (
                self.long_distance_conf_threshold
                if long_distanced_object
                else self.conf_threshold
            )
            self.tracker.offset = self.tracker.tracking_config["search_context"] if not long_distanced_object else self.tracker.tracking_config["search_context"] +0.5
        else:
            effective_threshold = 0.0

        # ── Occlusion entry ───────────────────────────────────────────────
        if score < effective_threshold and self.frame_idx >= 30 and self.enter_occlusion_on_loss:
            self.in_occlusion = True

            # ── Out-of-frame detection — multi-evidence, replaces old 4-line check ──
            is_exiting, exit_edge = self._detect_exit_direction(frame)
            self._out_of_frame = is_exiting
            self._exit_edge    = exit_edge

            loss_cause = self._classify_loss_cause()

            # Override loss_cause when we're confident the object left the frame —
            # shrinkage/drift skips are irrelevant for out-of-frame exits
            if is_exiting:
                loss_cause = 'out_of_frame'

            print(
                f"[occlusion entry] frame={self.frame_idx}  "
                f"loss_cause={loss_cause}  "
                f"out_of_frame={self._out_of_frame}  exit_edge={self._exit_edge}"
            )

            area_skip    = self._detect_shrinkage_onset(
                max_lookback  = self.shrinkage_max_lookback,
                min_drop_frac = self.shrinkage_min_drop_frac,
            )
            drift_skip   = self._detect_center_drift_skip(max_lookback=self.shrinkage_max_lookback)
            dynamic_skip = max(area_skip, drift_skip)

            # Skip is irrelevant for out-of-frame and camera motion exits
            effective_skip = 0 if loss_cause in ('camera_motion', 'out_of_frame') else dynamic_skip


            print(
                f"[occlusion entry] frame={self.frame_idx}  "
                f"loss_cause={loss_cause}  "
                f"dynamic_skip={dynamic_skip}  "
                f"effective_skip={effective_skip}  "
                f"history_len={len(self._conf_history)}"
            )

            # Reset phase state at every occlusion entry
            self._occ_phase          = 0
            self._pending_candidates = []

            self.ekf = self._rebuild_ekf_from_clean_history(skip_override=effective_skip)
            self.ekf.predict(H=self._last_H, H_reliable=self._last_H_reliable)
            self._init_search_centre_from_history(skip_override=effective_skip)
            self.tracker.dynamic_update = False
            self._occ_frames = 0
            self.tracker.disable_tta()
            return self._occlusion_update(frame)

        # ── Motion model DISABLED as output — use SiamABC bbox directly ──
        # EKF still runs (predict already done above, update below) to keep
        # its state accurate for the eventual occlusion-entry rebuild.
        self.ekf.update(pred_bbox)

        cam_disp = self._h_translation_magnitude(self._last_H, frame)
        self._cam_disp_history.append(cam_disp)

        h_fr, w_fr = frame.shape[:2]
        # if self._last_H is not None and self._last_H_reliable:
        #     cx, cy = w_fr / 2.0, h_fr / 2.0
        #     denom  = self._last_H[2,0]*cx + self._last_H[2,1]*cy + self._last_H[2,2] + 1e-8
        #     ncx    = (self._last_H[0,0]*cx + self._last_H[0,1]*cy + self._last_H[0,2]) / denom
        #     ncy    = (self._last_H[1,0]*cx + self._last_H[1,1]*cy + self._last_H[1,2]) / denom
        #     self._cam_vel_history.append(np.array([ncx - cx, ncy - cy]))
        # else:
        #     self._cam_vel_history.append(np.zeros(2))


        if self._last_H is not None:
            cx, cy = w_fr / 2.0, h_fr / 2.0
            denom  = self._last_H[2,0]*cx + self._last_H[2,1]*cy + self._last_H[2,2] + 1e-8
            ncx    = (self._last_H[0,0]*cx + self._last_H[0,1]*cy + self._last_H[0,2]) / denom
            ncy    = (self._last_H[1,0]*cx + self._last_H[1,1]*cy + self._last_H[1,2]) / denom
            self._cam_vel_history.append(np.array([ncx - cx, ncy - cy]))
        else:
            self._cam_vel_history.append(np.zeros(2))

        cx = float(pred_bbox[0] + pred_bbox[2] / 2.0)
        cy = float(pred_bbox[1] + pred_bbox[3] / 2.0)
        self._center_history.append(np.array([cx, cy]))

        # Velocity from finite differences (not EKF) — camera-compensated
        # because SiamABC output is in image coords (camera motion already
        # expressed as pixel movement).
        self.velocity = self._compute_velocity_from_history(pred_bbox)
        self._vel_history.append(self.velocity.copy())  


        desc = _extract_descriptor(frame, pred_bbox)
        if desc is not None:
            self.memory.try_admit(pred_bbox, desc, self.current_bbox)
        self.current_bbox = pred_bbox.copy()
        self.held_box     = pred_bbox.copy()
        self._size_history.append((int(pred_bbox[2]), int(pred_bbox[3])))
        # self._conf_history.append((pred_bbox.copy(), self.velocity.copy()))
        self._conf_history.append((pred_bbox.copy(), self.velocity.copy(), 
                           self._last_H, self._last_H_reliable))

        return pred_bbox, score

    # ── Velocity (finite-difference, used during normal tracking) ─────────────

    def _compute_velocity_from_history(
        self,
        current_bbox:  np.ndarray,
        lookback:      Optional[int]   = None,
        smooth_alpha:  Optional[float] = None,
    ) -> np.ndarray:
        """
        Finite-difference velocity from the last `lookback` stored positions,
        EMA-smoothed with the previous velocity estimate.

        Using raw SiamABC positions (not EKF) means the velocity includes
        camera motion expressed as pixel displacement — that's intentional here
        because the EKF will subtract it via homography when it takes over
        during occlusion.
        """
        lb    = lookback     if lookback     is not None else self.velocity_lookback
        alpha = smooth_alpha if smooth_alpha is not None else self.velocity_smooth_alpha

        if not self._conf_history:
            return np.zeros(2)

        history = list(self._conf_history)
        n_back  = min(lb, len(history))
        if n_back == 0:
            return np.zeros(2)

        curr_cx = current_bbox[0] + current_bbox[2] / 2.0
        curr_cy = current_bbox[1] + current_bbox[3] / 2.0

        ref_bbox = history[-n_back][0]
        ref_cx   = ref_bbox[0] + ref_bbox[2] / 2.0
        ref_cy   = ref_bbox[1] + ref_bbox[3] / 2.0

        raw_vx = (curr_cx - ref_cx) / n_back
        raw_vy = (curr_cy - ref_cy) / n_back

        vx = alpha * raw_vx + (1.0 - alpha) * self.velocity[0]
        vy = alpha * raw_vy + (1.0 - alpha) * self.velocity[1]
        return np.array([vx, vy])

    # ── Search-centre init at occlusion entry ─────────────────────────────────

    def _init_search_centre_from_history(self, skip_override=None) -> None:
        history = list(self._conf_history)
        skip    = (min(self.history_skip_last, len(history) - 1)
                   if skip_override is None else skip_override)
        clean   = history[:len(history) - skip] if skip > 0 else history

        if clean:
            last_clean_bbox  = clean[-1][0].astype(float)
            self._search_cx  = last_clean_bbox[0] + last_clean_bbox[2] / 2.0
            self._search_cy  = last_clean_bbox[1] + last_clean_bbox[3] / 2.0
        else:
            self._search_cx  = float(self.current_bbox[0] + self.current_bbox[2] / 2.0)
            self._search_cy  = float(self.current_bbox[1] + self.current_bbox[3] / 2.0)

    # ── Occlusion update — dispatcher ─────────────────────────────────────────

    def _occlusion_update(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        h_fr, w_fr = frame.shape[:2]

        ekf_raw         = self.ekf.get_bbox()
        self._search_cx = float(ekf_raw[0] + ekf_raw[2] / 2.0)
        self._search_cy = float(ekf_raw[1] + ekf_raw[3] / 2.0)
        self.held_box   = self._clamp_bbox_to_frame(ekf_raw, frame)
        self.velocity   = self.ekf.get_velocity()
        self._occ_frames += 1

        # ── When out-of-frame, pin search position to the exit edge ──────────
        # EKF position is now frozen (not drifting), but we explicitly pin the
        # search center to the frame boundary so ROI logic always hugs the edge
        # regardless of how stale the EKF state is.
        if self._out_of_frame and self._exit_edge is not None:
            if self._exit_edge == 'right':
                self._search_cx = float(w_fr - 1)
            elif self._exit_edge == 'left':
                self._search_cx = 0.0
            elif self._exit_edge == 'bottom':
                self._search_cy = float(h_fr - 1)
            elif self._exit_edge == 'top':
                self._search_cy = 0.0

        # ── Out-of-frame state management ────────────────────────────────────
        if self._out_of_frame:
            ekf_inside = (0 <= self._search_cx < w_fr and 0 <= self._search_cy < h_fr)
            vel_inward = False
            if ekf_inside and self._exit_edge is not None:
                vel_inward = {
                    'right':  float(self.velocity[0]) < 0,
                    'left':   float(self.velocity[0]) > 0,
                    'bottom': float(self.velocity[1]) < 0,
                    'top':    float(self.velocity[1]) > 0,
                }.get(self._exit_edge, True)
            if ekf_inside and vel_inward:
                self._out_of_frame = False
                self._exit_edge    = None
        else:
            if (self._search_cx < 0 or self._search_cx >= w_fr or
                    self._search_cy < 0 or self._search_cy >= h_fr):
                if   self._search_cx >= w_fr: self._exit_edge = 'right'
                elif self._search_cx < 0:     self._exit_edge = 'left'
                elif self._search_cy >= h_fr: self._exit_edge = 'bottom'
                else:                         self._exit_edge = 'top'
                self._out_of_frame = True

        if self._occ_phase == 0:
            return self._occ_phase_siam(frame)
        elif self._occ_phase == 1:
            return self._occ_phase_yolo(frame)
        else:
            return self._occ_phase_verify(frame)



    def _occ_phase_siam(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        rx, ry, rw, rh = self._get_yolo_search_roi(frame=frame)

        if self._size_history:
            obj_w = max(1, int(np.median([s[0] for s in self._size_history])))
            obj_h = max(1, int(np.median([s[1] for s in self._size_history])))
        else:
            obj_w = max(1, int(self.held_box[2]))
            obj_h = max(1, int(self.held_box[3]))

        # Seed the tracker at the ROI center with the expected object size
        roi_cx = rx + rw / 2.0
        roi_cy = ry + rh / 2.0
        seed_bbox = np.array([
            int(roi_cx - obj_w / 2.0),
            int(roi_cy - obj_h / 2.0),
            obj_w, obj_h,
        ], dtype=int)
        seed_bbox = self._clamp_bbox_to_frame(seed_bbox, frame)

        self.tracker.tracking_state.bbox = seed_bbox
        pred_bbox, score, _ = self.tracker.update(frame)
        pred_bbox = np.array(pred_bbox, dtype=int)

        if score >= self.reacq_threshold:
            pred_desc = _extract_descriptor(frame, pred_bbox)

            drm_results = self.memory.drm_match(
                frame           = frame,
                candidates      = [pred_bbox],
                ref_bbox        = self.held_box,
                velocity        = self.velocity,
                distractor_bank = self._distractor_bank,
                search_cx       = self._search_cx,
                search_cy       = self._search_cy,
                dist_sigma      = self._drm_dist_sigma_factor * max(
                                    int(self.held_box[2]), int(self.held_box[3])),
                **self._drm_kwargs,
            )

            drm_score = drm_results[0][1] if drm_results else -1.0
            drm_ok    = drm_score >= self.app_match_threshold

            print(f"[occ frame {self._occ_frames}] phase=Model  "
                  f"match={drm_score:.3f}  verify={None}  "
                  f"pass={drm_ok}")

            if drm_ok:
                self.recovered_early_occlusion = True
                return self._commit_reacquisition(frame, pred_bbox, pred_desc, score)
            self.tracker.tracking_state.bbox = self.held_box.copy()


        self._occ_phase = 1
        return self.held_box, score
    # ── Phase 1: YOLO detect + DRM match ─────────────────────────────────────

    def _occ_phase_yolo(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Run YOLO in the uncertainty-driven ROI and rank detections with the
        DRM matcher.  Ranked candidates are stored for the next frame (phase 2).

        If no detections pass DRM, the held_box is nudged toward the nearest
        detection (as before) and we skip phase 2 by jumping straight back to
        phase 0 to avoid wasting a frame on a vacuous verify step.
        """
        detections      = self._yolo_detect(frame)
        self._last_yolo = detections
        self._pending_candidates = []

        if detections:
            # Update distractor bank
            for det in detections:
                if _iou(det, self.held_box) >= self.tau_occ:
                    det_desc = _extract_descriptor(frame, det)
                    if det_desc is not None:
                        self._distractor_bank.append(det_desc)

            if self._size_history:
                obj_w = float(np.median([s[0] for s in self._size_history]))
                obj_h = float(np.median([s[1] for s in self._size_history]))
            else:
                obj_w = float(self.held_box[2])
                obj_h = float(self.held_box[3])
            dist_sigma = self._drm_dist_sigma_factor * max(obj_w, obj_h)

            self._pending_candidates = self.memory.drm_match(
                frame           = frame,
                candidates      = detections,
                ref_bbox        = self.held_box,
                velocity        = self.velocity,
                distractor_bank = self._distractor_bank,
                search_cx       = self._search_cx,
                search_cy       = self._search_cy,
                dist_sigma      = dist_sigma,
                **self._drm_kwargs,
            )

            print(f"[occ frame {self._occ_frames}] phase=yolo  "
                  f"detections={len(detections)}  "
                  f"ranked={len(self._pending_candidates)}  "
                  f"drm={self.memory.drm_size()}  ram={len(self.memory)}  "
                  f"uncertainty={self.ekf.get_uncertainty():.1f}px")

            if not self._pending_candidates:
                # No candidates passed DRM — nudge and skip straight to phase 0
                self.held_box = self._nudge_toward_nearest(frame, detections)
                self.ekf.nudge_position(self.held_box)
                self.tracker.tracking_state.bbox = self.held_box.copy()
                self._occ_phase = 0
                return self.held_box, 0.0

        # Have candidates → advance to verify phase
        self._occ_phase = 2 if self._pending_candidates else 0
        return self.held_box, 0.0

    # ── Phase 2: Tracker verification ────────────────────────────────────────

    def _occ_phase_verify(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Verify each DRM-ranked candidate with SiamABC.

        Candidates were detected on the previous frame, so each bbox is
        velocity-compensated by 1 frame before passing to run_track_for_candidate
        to account for object motion between the YOLO frame and this frame.

        Success → exit occlusion (_commit_reacquisition).
        All fail → reset to phase 0 (try SiamABC next frame).
        """
        if not self._pending_candidates:
            self._occ_phase = 0
            self.tracker.tracking_state.bbox = self.held_box.copy()
            return self.held_box, 0.0

        vx = float(self.velocity[0])
        vy = float(self.velocity[1])

        for match_bbox, match_score in self._pending_candidates:
            # 1-frame motion compensation: candidates are 1 frame old
            adjusted    = match_bbox.astype(float)
            adjusted[0] += vx
            adjusted[1] += vy
            adjusted    = np.array(adjusted, dtype=int)

            self.tracker.dynamic_update = False
            verify_bbox, verify_score, _ = self.tracker.run_track_for_candidate(
                frame, adjusted)
            verify_bbox = np.array(verify_bbox, dtype=int)

            print(f"[occ frame {self._occ_frames}] phase=verify  "
                  f"match={match_score:.3f}  verify={verify_score:.3f}  "
                  f"pass={verify_score >= self.reacq_threshold}")

            if verify_score >= self.reacq_threshold:
                self.recovered_early_occlusion=True
                desc = _extract_descriptor(frame, verify_bbox)
                return self._commit_reacquisition(frame, verify_bbox, desc, verify_score)

        # All candidates failed — back to SiamABC attempt next frame
        self._pending_candidates = []
        self._occ_phase = 0
        self.tracker.tracking_state.bbox = self.held_box.copy()
        return self.held_box, 0.0

    # ── Shared reacquisition exit ─────────────────────────────────────────────

    def _commit_reacquisition(
        self,
        frame: np.ndarray,
        bbox:  np.ndarray,
        desc:  Optional[np.ndarray],
        score: float,
    ) -> Tuple[np.ndarray, float]:
        """
        Single exit path shared by all three occlusion phases.
        Updates EKF with the accepted measurement, resets all occlusion state,
        and restores the tracker to normal-tracking mode.
        """
        self.ekf.update(bbox)
        ekf_bbox      = self.ekf.get_bbox()
        self.velocity = self.ekf.get_velocity()

        self.in_occlusion  = False
        self._out_of_frame = False
        self._exit_edge    = None
        self._occ_frames   = 0
        self._occ_phase    = 0
        self._pending_candidates = []
        self.tracker.enable_tta()
        self.tracker.dynamic_update = self.tracker.tracking_config["dynamic_update"]

        if desc is not None:
            self.memory.try_admit(ekf_bbox, desc, self.held_box)
        self.current_bbox = ekf_bbox.copy()
        self.held_box     = ekf_bbox.copy()
        self.tracker.tracking_state.bbox = ekf_bbox.copy()
        self._search_cx   = float(ekf_bbox[0] + ekf_bbox[2] / 2.0)
        self._search_cy   = float(ekf_bbox[1] + ekf_bbox[3] / 2.0)

        cx = float(ekf_bbox[0] + ekf_bbox[2] / 2.0)
        cy = float(ekf_bbox[1] + ekf_bbox[3] / 2.0)
        self._center_history.append(np.array([cx, cy]))
        cam_disp = np.zeros(2)
        if self._last_H is not None:
            h_fr, w_fr = frame.shape[:2]
            cx, cy = w_fr / 2.0, h_fr / 2.0
            denom  = self._last_H[2,0]*cx + self._last_H[2,1]*cy + self._last_H[2,2] + 1e-8
            ncx    = (self._last_H[0,0]*cx + self._last_H[0,1]*cy + self._last_H[0,2]) / denom
            ncy    = (self._last_H[1,0]*cx + self._last_H[1,1]*cy + self._last_H[1,2]) / denom
            cam_disp = np.array([ncx - cx, ncy - cy])
        self._cam_vel_history.append(cam_disp)
        
        self._conf_history.append((
            ekf_bbox.copy(),
            self.velocity.copy(),
            self._last_H,
            self._last_H_reliable,
        ))
        return ekf_bbox, score

    # ── Background homography estimation ─────────────────────────────────────

    # def _estimate_homography(
    #     self, frame: np.ndarray
    # ) -> Tuple[Optional[np.ndarray], bool, np.ndarray]:
    #     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    #     if self.prev_gray is None:
    #         return None, False, gray
    #     # else:
    #     #     return None, False, gray

    #     mask    = np.ones_like(self.prev_gray, dtype=np.uint8) * 255
    #     ref_box = self.held_box if self.held_box is not None else self.current_bbox
    #     if ref_box is not None:
    #         x, y, w, h = map(int, ref_box)
    #         pad        = max(10, int(max(w, h) * 0.15))
    #         y1b = max(0, y - pad);           y2b = min(mask.shape[0], y + h + pad)
    #         x1b = max(0, x - pad);           x2b = min(mask.shape[1], x + w + pad)
    #         mask[y1b:y2b, x1b:x2b] = 0

    #     step = 100
    #     y_g, x_g = np.mgrid[step//2:self.prev_gray.shape[0]:step,
    #                          step//2:self.prev_gray.shape[1]:step]
    #     grid_pts = np.vstack((x_g.flatten(), y_g.flatten())).T.astype(np.float32)
    #     valid_mask_vals = mask[grid_pts[:, 1].astype(int), grid_pts[:, 0].astype(int)]
    #     valid_pts = grid_pts[valid_mask_vals == 255]
    #     pts = valid_pts.reshape(-1, 1, 2)
    #     del mask

    #     H        = None
    #     reliable = False

    #     if pts is not None and len(pts) >= 8:
    #         new_pts, status, _ = cv2.calcOpticalFlowPyrLK(
    #             self.prev_gray, gray, pts, None)

    #         if status is not None:
    #             flat        = status.flatten() == 1
    #             good_old    = pts[flat]
    #             good_new    = new_pts[flat]
    #             del pts, new_pts, status, flat

    #             if len(good_old) >= 8:
    #                 H, inlier_mask = cv2.findHomography(
    #                     good_old, good_new,
    #                     cv2.RANSAC,
    #                     ransacReprojThreshold=3.0,
    #                 )
    #                 if H is not None and inlier_mask is not None:
    #                     inlier_ratio = float(inlier_mask.sum()) / len(good_old)
    #                     reliable     = (inlier_ratio >= self.homo_inlier_threshold)
    #                 del inlier_mask
    #             del good_old, good_new
    #         else:
    #             del pts, new_pts, status
    #     elif pts is not None:
    #         del pts

    #     return H, reliable, gray



    def _estimate_homography(
    self, frame: np.ndarray
) -> Tuple[Optional[np.ndarray], bool, np.ndarray]:
        """
        Optimisations vs. original
        ──────────────────────────
        1. Half-res optical flow   → 4× fewer pixels, biggest single win
        2. Cached grid points      → no alloc / mgrid every frame
        3. Vectorised bbox mask    → no uint8 mask array
        4. estimateAffinePartial2D → 4-DOF similarity RANSAC (vs 8-DOF homography)
                                    converges in ~4× fewer iterations
        5. Tighter LK params       → smaller window, fewer pyramid levels
        6. Capped RANSAC iters     → 500 instead of default 2 000
        """

        _LK_PARAMS = dict(
        winSize   = (20, 20),           # was default 21×21  → ~2.5× fewer ops/pt
        maxLevel  = 2,                  # was 3              → one less pyramid level
        criteria  = (
            cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
            8,                          # max iterations (was 20)
            0.04,                       # epsilon
        ),
    )
        SCALE = 0.5  # 0.5 recommended; tune to 0.4 for even more speed

        # ── 1. Downsample + greyscale ────────────────────────────────────────────
        small = cv2.resize(frame, None, fx=SCALE, fy=SCALE,
                        interpolation=cv2.INTER_LINEAR)
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        if self.prev_gray is None:          # prev_gray is now stored at SCALE res
            return None, False, gray
        

        if self.prev_gray.shape != gray.shape:
            prev_gray_scaled = cv2.resize(self.prev_gray, (gray.shape[1], gray.shape[0]),
                                        interpolation=cv2.INTER_LINEAR)
        else:
            prev_gray_scaled = self.prev_gray

        H        = None
        reliable = False

        # ── 2. Build grid once; reuse until resolution changes ───────────────────
        if self._cached_shape != gray.shape:
            step = 50                       # ≡ step=100 at full res when SCALE=0.5
            yg, xg = np.mgrid[
                step // 2 : gray.shape[0] : step,
                step // 2 : gray.shape[1] : step,
            ]
            self._cached_pts   = np.column_stack(
                (xg.ravel(), yg.ravel())
            ).astype(np.float32)
            self._cached_shape = gray.shape

        grid = self._cached_pts             # shape (N, 2)

        # ── 3. Exclude bbox region – fully vectorised, no mask array ─────────────
        ref_box = self.held_box if self.held_box is not None else self.current_bbox
        if ref_box is not None:
            x, y, w, h = (v * SCALE for v in map(int, ref_box))
            pad         = max(4, int(max(w, h) * 0.15))
            inside = (
                (grid[:, 0] >= x - pad) & (grid[:, 0] < x + w + pad) &
                (grid[:, 1] >= y - pad) & (grid[:, 1] < y + h + pad)
            )
            pts = grid[~inside].reshape(-1, 1, 2)
        else:
            pts = grid.reshape(-1, 1, 2)

        if len(pts) < 6:
            return H, reliable, gray

        # ── 4. Sparse optical flow ───────────────────────────────────────────────
        new_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray_scaled, gray, pts, None, **_LK_PARAMS
        )

        if status is None:
            return H, reliable, gray

        ok       = status.ravel() == 1
        good_old = pts[ok]
        good_new = new_pts[ok]

        if len(good_old) < 6:
            return H, reliable, gray

        # ── 5. Similarity transform (4 DOF) via RANSAC ───────────────────────────
        #    Handles pan / tilt / zoom / roll – sufficient for most cameras.
        #    ~4× faster RANSAC than findHomography (8 DOF) in practice.
        A, inliers = cv2.estimateAffinePartial2D(
            good_old, good_new,
            method               = cv2.RANSAC,
            ransacReprojThreshold= 3.0,
            maxIters             = 500,     # default 2 000 – overkill for grid pts
            confidence           = 0.99,
        )

        if A is not None and inliers is not None:
            inlier_ratio = float(inliers.sum()) / len(good_old)
            reliable     = inlier_ratio >= self.homo_inlier_threshold

            # Promote 2×3 affine → 3×3 homography; rescale translation to full-res
            H = np.eye(3, dtype=np.float64)
            H[:2, :] = A
            H[0, 2] /= SCALE
            H[1, 2] /= SCALE

        return H, reliable, gray   # gray is at SCALE res → store as self.prev_gray

    # ── Nudge toward nearest detection ───────────────────────────────────────

    def _nudge_toward_nearest(self, frame: np.ndarray,
                               detections: List[np.ndarray]) -> np.ndarray:
        ref  = self.memory.best_descriptor()
        hcx  = self.held_box[0] + self.held_box[2] / 2.0
        hcy  = self.held_box[1] + self.held_box[3] / 2.0

        best_det, best_rank = None, float('inf')
        for det in detections:
            dcx  = det[0] + det[2] / 2.0
            dcy  = det[1] + det[3] / 2.0
            dist = np.hypot(dcx - hcx, dcy - hcy)
            if ref is not None:
                desc = _extract_descriptor(frame, det)
                sim  = _cos_sim(ref, desc) if desc is not None else 0.0
                rank = dist * (1.0 - 0.5 * sim)
            else:
                rank = dist
            if rank < best_rank:
                best_rank, best_det = rank, det

        if best_det is None:
            return self.held_box.copy()

        dcx    = best_det[0] + best_det[2] / 2.0
        dcy    = best_det[1] + best_det[3] / 2.0
        new_cx = hcx + self.nudge_alpha * (dcx - hcx)
        new_cy = hcy + self.nudge_alpha * (dcy - hcy)

        hw, hh   = self.held_box[2], self.held_box[3]
        h_fr, w_fr = frame.shape[:2]
        nx = int(np.clip(new_cx - hw / 2.0, 0, w_fr - 1))
        ny = int(np.clip(new_cy - hh / 2.0, 0, h_fr - 1))
        return np.array([nx, ny,
                         int(np.clip(hw, 1, w_fr - nx)),
                         int(np.clip(hh, 1, h_fr - ny))], dtype=int)

    # ── YOLO search ROI ───────────────────────────────────────────────────────

    # def _get_yolo_search_roi(self, frame: np.ndarray) -> Tuple[int, int, int, int]:
    #     h_fr, w_fr = frame.shape[:2]
    #     max_side = min(w_fr, h_fr) // 2 

    #     if self.frame_idx <= 30 or self.recovered_early_occlusion==False:
    #         self.recovered_early_occlusion=False
    #         scale = 0.4

    #         bw = int(w_fr * scale)
    #         bh = int(h_fr * scale)

    #         x = (w_fr - bw) // 2
    #         y = (h_fr - bh) // 2

    #         return x, y, bw, bh

    #     if self._size_history:
    #         obj_w = max(1, int(np.median([s[0] for s in self._size_history])))
    #         obj_h = max(1, int(np.median([s[1] for s in self._size_history])))
    #     else:
    #         obj_w = max(1, int(self.held_box[2]))
    #         obj_h = max(1, int(self.held_box[3]))
    #     obj_size = (obj_w + obj_h) // 2

    #     steps       = self._occ_frames // max(1, self.search_expand_growth_every)
    #     time_expand = float(self.search_expand_growth_factor ** steps)

    #     effective_expand = min(
    #         self.roi_start_expand * time_expand, self.yolo_search_expand
    #     )

    #     if self._out_of_frame and self._exit_edge is not None:
    #         side = max(1, int(obj_size * effective_expand))

    #         if self._exit_edge == 'right':
    #             dist_from_edge = self._search_cx - w_fr
    #             if dist_from_edge > obj_w * 2:
    #                 return 0, 0, 0, 0
    #             scy = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
    #             x1  = max(0, w_fr - side)
    #             y1  = int(scy - side // 2)
    #             return x1, y1, max(1, w_fr - x1), side

    #         elif self._exit_edge == 'left':
    #             scy = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
    #             y1  = int(scy - side // 2)
    #             return 0, y1, side, side

    #         elif self._exit_edge == 'bottom':
    #             scx = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
    #             x1  = int(scx - side // 2)
    #             y1  = max(0, h_fr - side)
    #             return x1, y1, side, max(1, h_fr - y1)

    #         else:  # top
    #             scx = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
    #             x1  = int(scx - side // 2)
    #             return x1, 0, side, side

    #     half = (obj_size * effective_expand) / 2.0
    #     half = min(half, max_side / 2.0)


    #     if half * 2 >= min(w_fr, h_fr):
    #         side = min(w_fr, h_fr)
    #         cx   = int(np.clip(self._search_cx, side // 2, w_fr - side // 2))
    #         cy   = int(np.clip(self._search_cy, side // 2, h_fr - side // 2))
    #         return cx - side // 2, cy - side // 2, side, side

    #     scx  = float(np.clip(self._search_cx, half, w_fr - half))
    #     scy  = float(np.clip(self._search_cy, half, h_fr - half))
    #     side = int(half * 2)
    #     return int(scx - half), int(scy - half), max(1, side), max(1, side)
    

    def _get_yolo_search_roi(self, frame: np.ndarray) -> Tuple[int, int, int, int]:
        h_fr, w_fr = frame.shape[:2]
        # max_side = min(w_fr, h_fr) // 2
        max_side = min(w_fr, h_fr)

        if self.frame_idx <= 30 or self.recovered_early_occlusion == False:
            self.recovered_early_occlusion = False
            scale = 0.4
            bw = int(w_fr * scale)
            bh = int(h_fr * scale)
            x  = (w_fr - bw) // 2
            y  = (h_fr - bh) // 2
            return x, y, bw, bh

        if self._size_history:
            obj_w = max(1, int(np.median([s[0] for s in self._size_history])))
            obj_h = max(1, int(np.median([s[1] for s in self._size_history])))
        else:
            obj_w = max(1, int(self.held_box[2]))
            obj_h = max(1, int(self.held_box[3]))
        obj_size = (obj_w + obj_h) // 2

        steps            = self._occ_frames // max(1, self.search_expand_growth_every)
        time_expand      = float(self.search_expand_growth_factor ** steps)
        effective_expand = min(self.roi_start_expand * time_expand, self.yolo_search_expand)

        if self._out_of_frame and self._exit_edge is not None:
            side = max(1, int(obj_size * effective_expand))

            if self._exit_edge == 'right':
                if (self._search_cx - w_fr) > obj_w * 2:
                    return 0, 0, 0, 0
                scy = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
                x1, y1 = w_fr - side, int(scy - side // 2)
                rw, rh = side, side

            elif self._exit_edge == 'left':
                scy = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
                x1, y1 = 0, int(scy - side // 2)
                rw, rh = side, side

            elif self._exit_edge == 'bottom':
                scx = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
                x1, y1 = int(scx - side // 2), h_fr - side
                rw, rh = side, side

            else:  # top
                scx = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
                x1, y1 = int(scx - side // 2), 0
                rw, rh = side, side

        else:  # ← THIS WAS MISSING
            half  = min((obj_size * effective_expand) / 2.0, max_side / 2.0)
            cx_lo = min(half, w_fr / 2.0)
            cy_lo = min(half, h_fr / 2.0)
            scx   = float(np.clip(self._search_cx, cx_lo, w_fr - cx_lo))
            scy   = float(np.clip(self._search_cy, cy_lo, h_fr - cy_lo))
            side  = max(1, int(half * 2))
            x1, y1 = int(scx - half), int(scy - half)
            rw, rh = side, side

        # Single final clamp
        x1 = max(0, x1)
        y1 = max(0, y1)
        rw = min(rw, w_fr - x1)
        rh = min(rh, h_fr - y1)
        rw = max(1, rw)
        rh = max(1, rh)
        return x1, y1, rw, rh
    # ── YOLO (with per-frame cache) ───────────────────────────────────────────

    def _yolo_detect(self, frame: np.ndarray) -> List[np.ndarray]:
        rx, ry, rw, rh = self._get_yolo_search_roi(frame)
        crop           = frame[ry:ry + rh, rx:rx + rw]
        if crop.size == 0:
            self._yolo_cache = []
            return []
        results = self.yolo.predict(crop, conf=self.yolo_conf,
                                    iou=self.yolo_iou_thr, verbose=False, imgsz=320)
        boxes = []
        if results and results[0].boxes is not None:
            for xyxy in results[0].boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = xyxy
                boxes.append(np.array([int(x1) + rx, int(y1) + ry,
                                       int(x2 - x1), int(y2 - y1)], dtype=int))
        self._yolo_cache = boxes
        return boxes

    def _yolo_detect_cached(self, frame: np.ndarray) -> List[np.ndarray]:
        if self._yolo_cache:
            return self._yolo_cache
        return self._yolo_detect(frame)

    # ── Misc helpers ──────────────────────────────────────────────────────────

    def _clamp_bbox_to_frame(self, bbox: np.ndarray, frame: np.ndarray) -> np.ndarray:
        h_fr, w_fr = frame.shape[:2]
        x, y, w, h = bbox
        w = int(np.clip(w, 1, w_fr))
        h = int(np.clip(h, 1, h_fr))
        x = int(np.clip(x, -w, w_fr))
        y = int(np.clip(y, -h, h_fr))
        return np.array([x, y, w, h], dtype=int)

    def _h_translation_magnitude(self, H, frame: np.ndarray) -> float:
        if H is None:
            return 0.0
        h, w   = frame.shape[:2]
        cx, cy = w / 2.0, h / 2.0
        denom  = H[2, 0]*cx + H[2, 1]*cy + H[2, 2] + 1e-8
        new_cx = (H[0, 0]*cx + H[0, 1]*cy + H[0, 2]) / denom
        new_cy = (H[1, 0]*cx + H[1, 1]*cy + H[1, 2]) / denom
        return float(np.hypot(new_cx - cx, new_cy - cy))

    def _classify_loss_cause(
        self,
        cam_disp_threshold:   float = 18.0,
        area_shrink_threshold: float = -0.005,
    ) -> str:
        history  = list(self._conf_history)
        cam_hist = list(self._cam_disp_history)

        area_vote = 'occlusion'
        if len(history) >= 3:
            areas      = np.array([float(b[2] * b[3]) for b, _ , _ , _ in history])
            med_area   = float(np.median(areas))
            n          = len(areas)
            t          = np.arange(n, dtype=float)
            slope      = float(np.polyfit(t, areas, 1)[0])
            norm_slope = slope / (med_area + 1e-6)
            if norm_slope >= area_shrink_threshold:
                area_vote = 'camera_motion'

        cam_vote = 'occlusion'
        if cam_hist:
            weights   = np.exp(np.linspace(-1, 0, len(cam_hist)))
            weights  /= weights.sum()
            mean_disp = float(np.dot(weights, cam_hist))
            if mean_disp >= cam_disp_threshold:
                cam_vote = 'camera_motion'

        if area_vote == 'camera_motion' and cam_vote == 'camera_motion':
            return 'camera_motion'
        return 'occlusion'

    def _is_long_distance(self, frame: np.ndarray) -> bool:
        if self.long_distance_mode:
            return True
        if not self._size_history:
            return False
        h_fr, w_fr  = frame.shape[:2]
        frame_area  = float(h_fr * w_fr)
        obj_w       = float(np.median([s[0] for s in self._size_history]))
        obj_h       = float(np.median([s[1] for s in self._size_history]))
        obj_area    = obj_w * obj_h
        return (obj_area / (frame_area + 1e-8)) < self.long_distance_area_fraction

    def _rebuild_ekf_from_clean_history(self, skip_override=None) -> BBoxEKF:
        history = list(self._conf_history)
        raw_skip = self.history_skip_last if skip_override is None else skip_override
        skip     = min(raw_skip, max(0, len(history) - 2))
        clean    = history[:len(history) - skip] if skip > 0 else history

        if len(clean) == 0:
            return self.ekf

        first_bbox, _, _, _ = clean[0]
        fresh_ekf = BBoxEKF(
            first_bbox,
            process_noise=self.ekf_process_noise,
            meas_noise=self.ekf_meas_noise,
        )

        for (bbox, _vel, h, h_rel) in clean[1:]:
            fresh_ekf.predict(H=h, H_reliable=(h is not None))
            fresh_ekf.update(bbox)

        if False:
            # Long enough history → trust EKF replay velocity entirely
            fresh_ekf.P[2, 2] = 30.0
            fresh_ekf.P[3, 3] = 30.0
        else:
            # Short history → override with robust windowed estimate
            robust_vel = self._robust_velocity_from_history(skip=skip ,window=self.velocity_window_average,
                                   decay=0.97, clip_percentile=96.0)
            fresh_ekf.x[2] = float(robust_vel[0])
            fresh_ekf.x[3] = float(robust_vel[1])
            fresh_ekf.P[2, 2] = 40.0
            fresh_ekf.P[3, 3] = 40.0

        return fresh_ekf


    def _detect_shrinkage_onset(
        self,
        max_lookback:  int   = 10,
        min_drop_frac: float = 0.002,
        smooth_k:      int   = 3,
    ) -> int:
        """
        Retrospectively find how many trailing conf_history frames are
        corrupted by pre-occlusion bbox shrinkage.

        Returns the skip count (0 = no shrinkage detected).
        Tune min_drop_frac per scenario:
          - Close-range large objects → 0.08–0.15  (natural size variation is high)
          - General purpose           → 0.04–0.06
          - Distant small objects     → 0.002–0.03
        """
        history = list(self._conf_history)
        n = len(history)
        if n < 4:
            return 0

        areas = np.array([float(b[2] * b[3]) for b, _ , _ , _ in history], dtype=float)

        smoothed = np.array([
            float(np.median(areas[max(0, i - smooth_k + 1): i + 1]))
            for i in range(n)
        ])

        # 95th-percentile reference is robust when shrinkage spans >30% of history
        ref_area = float(np.percentile(smoothed, 95))
        if ref_area <= 0:
            return 0

        threshold = ref_area * (1.0 - min_drop_frac)

        skip       = 0
        gap_budget = 3
        gaps_used  = 0

        for i in range(n - 1, max(n - 1 - max_lookback, -1), -1):
            if smoothed[i] < threshold:
                skip      += 1
                gaps_used  = 0
            elif skip > 0 and gaps_used < gap_budget:
                skip      += 1
                gaps_used += 1
            else:
                break

        if skip >= max_lookback:
            full_drop = (ref_area - float(np.mean(smoothed[n - skip:]))) / (ref_area + 1e-6)
            if full_drop < min_drop_frac:
                return 0

        return max(skip , 5) 
    

    def _detect_center_drift_skip(
    self,
    max_lookback: int   = 20,
    spike_factor: float = 2.5,
) -> int:
        """
        Scan backwards through _center_history for frames where per-frame
        speed spikes above spike_factor * median of the stable history.
        Returns how many trailing frames to skip.
        """
        hist = list(self._center_history)
        if len(hist) < 4:
            return 0

        centers = np.stack(hist)                                  # (n, 2)
        speeds  = np.linalg.norm(np.diff(centers, axis=0), axis=1)  # (n-1,)
        n       = len(speeds)

        # Reference: median speed of the first 2/3 of history (stable period)
        ref_n   = max(2, n * 2 // 3)
        ref_mag = float(np.median(speeds[:ref_n])) + 1e-6
        thresh  = ref_mag * spike_factor

        skip      = 0
        gap_budget = 2   # tolerate brief dips (same pattern as shrinkage detector)
        gaps_used  = 0

        for i in range(n - 1, max(n - 1 - max_lookback, -1), -1):
            if speeds[i] > thresh:
                skip      += 1
                gaps_used  = 0
            elif skip > 0 and gaps_used < gap_budget:
                skip      += 1
                gaps_used += 1
            else:
                break

        return skip

    # ── Properties ───────────────────────────────────────────────────────────


    def _detect_exit_direction(
    self,
    frame: np.ndarray,
    lookahead_frames: int = 6,
    trend_frames:     int = 10,
    margin_factor:    float = 0.5,
) -> Tuple[bool, Optional[str]]:
        """
        Determine whether the object is exiting / has exited the frame,
        and which edge it used.  Uses three independent evidence sources:

        1. Last bbox edge proximity + velocity direction (instant signal)
        2. Velocity extrapolation N frames ahead (catches fast exits)
        3. Linear trajectory trend over recent history (catches gradual exits
            where neither bbox nor velocity alone is conclusive)

        Returns (is_exiting: bool, exit_edge: Optional[str]).
        """
        h_fr, w_fr = frame.shape[:2]

        last_bbox = self.current_bbox.astype(float)
        lx1, ly1, lw, lh = last_bbox
        lx2, ly2 = lx1 + lw, ly1 + lh
        lcx = lx1 + lw / 2.0
        lcy = ly1 + lh / 2.0

        vx = float(self.velocity[0])
        vy = float(self.velocity[1])

        # ── Evidence 1: bbox edge proximity + velocity direction ──────────────
        margin = max(lw, lh) * margin_factor
        prox_right  = lx2 > w_fr - margin and vx > 0
        prox_left   = lx1 < margin         and vx < 0
        prox_bottom = ly2 > h_fr - margin  and vy > 0
        prox_top    = ly1 < margin         and vy < 0

        # ── Evidence 2: velocity extrapolation ───────────────────────────────
        fut_cx = lcx + vx * lookahead_frames
        fut_cy = lcy + vy * lookahead_frames
        extrap_right  = fut_cx >= w_fr
        extrap_left   = fut_cx < 0
        extrap_bottom = fut_cy >= h_fr
        extrap_top    = fut_cy < 0

        # ── Evidence 3: linear trajectory trend ──────────────────────────────
        history = list(self._conf_history)
        n_use   = min(trend_frames, len(history))
        trend_right = trend_left = trend_bottom = trend_top = False

        if n_use >= 3:
            recent   = history[-n_use:]
            xs = np.array([float(b[0] + b[2] / 2) for b, *_ in recent])
            ys = np.array([float(b[1] + b[3] / 2) for b, *_ in recent])
            t  = np.arange(n_use, dtype=float)

            vx_trend = float(np.polyfit(t, xs, 1)[0])
            vy_trend = float(np.polyfit(t, ys, 1)[0])

            fut_tx = lcx + vx_trend * lookahead_frames
            fut_ty = lcy + vy_trend * lookahead_frames
            trend_right  = fut_tx >= w_fr  and vx_trend > 0
            trend_left   = fut_tx < 0      and vx_trend < 0
            trend_bottom = fut_ty >= h_fr  and vy_trend > 0
            trend_top    = fut_ty < 0      and vy_trend < 0

        # ── Combine: any two independent signals fire → confident exit ────────
        def _votes(right, left, bottom, top):
            return {
                'right':  right,
                'left':   left,
                'bottom': bottom,
                'top':    top,
            }

        e1 = _votes(prox_right,   prox_left,   prox_bottom,   prox_top)
        e2 = _votes(extrap_right, extrap_left, extrap_bottom, extrap_top)
        e3 = _votes(trend_right,  trend_left,  trend_bottom,  trend_top)

        for edge in ('right', 'left', 'bottom', 'top'):
            evidence_count = sum([e1[edge], e2[edge], e3[edge]])
            if evidence_count >= 2:            # ≥2 of 3 signals agree
                return True, edge

        # Single strong signal: bbox already outside frame boundary
        if lx2 >= w_fr: return True, 'right'
        if lx1 <= 0:    return True, 'left'
        if ly2 >= h_fr: return True, 'bottom'
        if ly1 <= 0:    return True, 'top'

        return False, None


    def _robust_velocity_from_history(self, skip=0, window=80,
                                   decay=0.97, clip_percentile=80.0) -> np.ndarray:
        hist = list(self._center_history)
        if len(hist) < 2:
            return self.velocity.copy()

        conf_len = len(list(self._conf_history))
        hist = hist[-conf_len:] if len(hist) > conf_len else hist

        if skip > 0:
            hist = hist[:max(2, len(hist) - skip)]

        hist = hist[-window:] if len(hist) > window else hist
        if len(hist) < 2:
            return self.velocity.copy()

        centers    = np.stack(hist)
        frame_vels = np.diff(centers, axis=0)   # shape (n, 2)
        n          = len(frame_vels)

        weights    = np.array([decay ** (n - 1 - i) for i in range(n)], dtype=float)
        weights   /= weights.sum()

        avg = (weights[:, None] * frame_vels).sum(axis=0)

        # ── Camera compensation: same window, same weights ────────────────────
        cam_hist = list(self._cam_vel_history)
        # Align cam_hist to the same tail as center_history diffs
        # frame_vels[i] = center[i+1] - center[i], so we need cam_vel entries
        # that correspond to the same frames. cam_vel_history is appended in
        # _normal_update in the same order as _center_history, so we just take
        # the last n entries (after the same skip).
        cam_hist = cam_hist[-conf_len:] if len(cam_hist) > conf_len else cam_hist
        if skip > 0:
            cam_hist = cam_hist[:max(2, len(cam_hist) - skip)]
        cam_hist = cam_hist[-window:] if len(cam_hist) > window else cam_hist

        if len(cam_hist) >= n:
            # Take the last n entries to align with frame_vels
            cam_vels    = np.stack(cam_hist[-n:])           # shape (n, 2)
            avg_cam     = (weights[:, None] * cam_vels).sum(axis=0)
            avg        -= avg_cam

        # # If cam_hist is too short, skip compensation rather than misalign



        magnitudes = np.linalg.norm(frame_vels, axis=1)
        mag_cap    = float(np.percentile(magnitudes, clip_percentile))
        avg_mag    = float(np.linalg.norm(avg))
        if avg_mag > mag_cap and avg_mag > 0:
            avg = avg * (mag_cap / avg_mag)

        return avg

    @property
    def running_dynamic_bbox(self):
        return self.tracker.running_dynamic_bbox

    @property
    def running_dynamic_image(self):
        return self.tracker.running_dynamic_image
 
    @property
    def tracking_config(self):
        return self.tracker.tracking_config

    @property
    def tracking_state(self):
        return self.tracker.tracking_state

In [9]:
#TODO: add more recovery options: motion of target 
#TODO: make system slower to recover but more accurate.
#TODO: make system slower to enter occlusion but more robust to noise (e.g. require multiple consecutive low-confidence frames before declaring occlusion)


In [6]:
import cv2
import numpy as np
from collections import deque
from typing import List, Optional, Tuple
from ultralytics import YOLO


# ─────────────────────────────────────────────────────────────────────────────
# Helpers  (UNCHANGED)
# ─────────────────────────────────────────────────────────────────────────────

def _extract_descriptor(frame: np.ndarray, bbox, size=16) -> Optional[np.ndarray]:
    """phi(I_t, b) = normalised concat of grayscale patch + HSV histogram."""
    x, y, w, h = map(int, bbox)
    x, y = max(0, x), max(0, y)
    w, h = max(1, w), max(1, h)
    patch = frame[y:y+h, x:x+w]
    if patch.size == 0:
        return None
    gray   = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    p      = cv2.resize(gray, (size, size)).flatten().astype(np.float32)
    hsv    = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 4, 4],
                          [0, 180, 0, 256, 0, 256]).flatten().astype(np.float32)
    desc = np.concatenate([p, h_hist])
    norm = np.linalg.norm(desc)
    return desc / (norm + 1e-8)


def _iou(a, b) -> float:
    ax2, ay2 = a[0] + a[2], a[1] + a[3]
    bx2, by2 = b[0] + b[2], b[1] + b[3]
    ix1 = max(a[0], b[0]);  iy1 = max(a[1], b[1])
    ix2 = min(ax2,  bx2);   iy2 = min(ay2,  by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / (union + 1e-8)


def _cos_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


# ─────────────────────────────────────────────────────────────────────────────
# BBoxEKF  (UNCHANGED)
# ─────────────────────────────────────────────────────────────────────────────

class BBoxEKF:
    """
    Center-state EKF.  Internal state is [cx, cy, vx, vy].
    Corners are only used at input (bbox → center) and output (center → bbox).
    This makes velocity completely immune to bbox shrinkage at occlusion entry.
    """

    DIM_X = 4
    DIM_Z = 2   # we observe (cx, cy) only

    def __init__(self, bbox, process_noise: float = 2.0, meas_noise: float = 5.0,
                 vel_window: int = 100):
        x1, y1, w, h = bbox
        cx = float(x1 + w / 2.0)
        cy = float(y1 + h / 2.0)
        self._bw = float(w)
        self._bh = float(h)

        self.x = np.array([cx, cy, 0., 0.])
        self.P = np.diag([10., 10., 100., 100.]).astype(float)
        self.Q = np.diag([
            process_noise,
            process_noise,
            process_noise * 4,
            process_noise * 4,
        ]).astype(float)
        self.R = np.eye(self.DIM_Z, dtype=float) * meas_noise
        self._H_mat = np.array([
            [1., 0., 0., 0.],
            [0., 1., 0., 0.],
        ], dtype=float)

    def _compute_jacobian(self, H, H_reliable, cx, cy, new_cx_h, new_cy_h) -> np.ndarray:
        F = np.eye(self.DIM_X, dtype=float)
        F[0, 2] = 1.0
        F[1, 3] = 1.0
        if H is not None and H_reliable:
            denom = H[2, 0] * cx + H[2, 1] * cy + H[2, 2] + 1e-8
            A = (H[0, 0] - new_cx_h * H[2, 0]) / denom
            B = (H[0, 1] - new_cx_h * H[2, 1]) / denom
            C = (H[1, 0] - new_cy_h * H[2, 0]) / denom
            D = (H[1, 1] - new_cy_h * H[2, 1]) / denom
            F[0, 0] = A;   F[0, 1] = B
            F[1, 0] = C;   F[1, 1] = D
        return F

    def predict(self, H: Optional[np.ndarray] = None, H_reliable: bool = False):
        cx, cy = self.x[0], self.x[1]
        if H is not None and H_reliable:
            denom    = H[2, 0] * cx + H[2, 1] * cy + H[2, 2]
            new_cx_h = (H[0, 0] * cx + H[0, 1] * cy + H[0, 2]) / (denom + 1e-8)
            new_cy_h = (H[1, 0] * cx + H[1, 1] * cy + H[1, 2]) / (denom + 1e-8)
        else:
            new_cx_h = cx
            new_cy_h = cy
        self.x[0] = new_cx_h + self.x[2]
        self.x[1] = new_cy_h + self.x[3]
        F      = self._compute_jacobian(H, H_reliable, cx, cy, new_cx_h, new_cy_h)
        self.P = F @ self.P @ F.T + self.Q

    def update(self, bbox):
        x1, y1, w, h = bbox
        cx = float(x1 + w / 2.0)
        cy = float(y1 + h / 2.0)
        self._bw = 0.85 * self._bw + 0.15 * float(w)
        self._bh = 0.85 * self._bh + 0.15 * float(h)
        z     = np.array([cx, cy])
        innov = z - self._H_mat @ self.x
        S     = self._H_mat @ self.P @ self._H_mat.T + self.R
        K     = self.P @ self._H_mat.T @ np.linalg.inv(S)
        self.x     = self.x + K @ innov
        I_KH       = np.eye(self.DIM_X) - K @ self._H_mat
        self.P     = I_KH @ self.P

    def get_bbox(self) -> np.ndarray:
        cx, cy = self.x[0], self.x[1]
        x1 = cx - self._bw / 2.0
        y1 = cy - self._bh / 2.0
        return np.array([int(x1), int(y1),
                         max(1, int(self._bw)),
                         max(1, int(self._bh))], dtype=int)

    def get_velocity(self) -> np.ndarray:
        return self.x[2:4].copy()

    def get_uncertainty(self) -> float:
        return float(np.sqrt(np.diag(self.P[:2]).mean()))

    def reseed(self, bbox, velocity: np.ndarray):
        x1, y1, w, h = bbox
        self.x[0] = float(x1 + w / 2.0)
        self.x[1] = float(y1 + h / 2.0)
        self.x[2] = float(velocity[0])
        self.x[3] = float(velocity[1])
        self._bw  = float(w)
        self._bh  = float(h)
        self.P    = np.diag([25., 25., 40., 40.]).astype(float)

    def nudge_position(self, bbox):
        x1, y1, w, h = bbox
        self.x[0] = float(x1 + w / 2.0)
        self.x[1] = float(y1 + h / 2.0)
        self._bw  = float(w)
        self._bh  = float(h)


# ─────────────────────────────────────────────────────────────────────────────
# AppearanceMemory  (UNCHANGED)
# ─────────────────────────────────────────────────────────────────────────────

class AppearanceMemory:
    def __init__(
        self,
        capacity:     int   = 20,
        tau_iou:      float = 0.40,
        tau_area:     float = 0.25,
        drm_capacity: int   = 10,
        tau_sim:      float = 0.85,
        window_W:     int   = 5,
        mmin:         int   = 3,
    ):
        self.capacity = capacity
        self.tau_iou  = tau_iou
        self.tau_area = tau_area
        self._buf: deque = deque(maxlen=capacity)
        self.drm_capacity = drm_capacity
        self.tau_sim      = tau_sim
        self.window_W     = window_W
        self.mmin         = mmin
        self._drm: deque  = deque(maxlen=drm_capacity)
        self._t:   int    = 0

    def reset(self):
        self._buf.clear()
        self._drm.clear()
        self._t = 0

    def try_admit(self, bbox, desc: np.ndarray, prev_bbox) -> bool:
        iou_ok = _iou(bbox, prev_bbox) >= self.tau_iou
        if self._buf:
            med     = float(np.median([e[0][2] * e[0][3] for e in self._buf]))
            area_ok = abs(bbox[2] * bbox[3] - med) / (med + 1e-6) <= self.tau_area
        else:
            area_ok = True
        if iou_ok and area_ok:
            self._buf.append((np.array(bbox, dtype=int), desc.copy()))
            self._t += 1
            self._try_promote_to_drm(bbox, desc)
            return True
        return False

    def _try_promote_to_drm(self, bbox, new_desc: np.ndarray) -> None:
        recent = list(self._buf)[-self.window_W:]
        if len(recent) < self.mmin:
            return
        agreements = sum(
            1 for (_, d) in recent
            if _cos_sim(new_desc, d) >= self.tau_sim
        )
        if agreements >= self.mmin:
            self._drm.append((
                np.array(bbox, dtype=int),
                new_desc.copy(),
                self._t,
            ))

    def best_descriptor(self) -> Optional[np.ndarray]:
        return self._buf[-1][1] if self._buf else None

    def match(self, frame: np.ndarray, candidates: List[np.ndarray],
              threshold: float) -> Tuple[Optional[np.ndarray], float]:
        ref = self.best_descriptor()
        if ref is None or not candidates:
            return None, -1.0
        best_box, best_score = None, -1.0
        for bbox in candidates:
            desc = _extract_descriptor(frame, bbox)
            if desc is None:
                continue
            s = _cos_sim(ref, desc)
            if s > best_score:
                best_score, best_box = s, bbox
        if best_score >= threshold:
            return np.array(best_box, dtype=int), best_score
        return None, best_score

    def drm_match(
        self,
        frame:            np.ndarray,
        candidates:       List[np.ndarray],
        ref_bbox:         np.ndarray,
        velocity:         np.ndarray,
        distractor_bank,
        lam_iou:          float = 0.40,
        lam_app:          float = 0.30,
        lam_mot:          float = 0.20,
        lam_time:         float = 0.10,
        alpha:            float = 0.05,
        gamma:            float = 0.25,
        margin:           float = 0.35,
        top_k:            int   = 3,
        skip_threshold:   float = 0.80,
        search_cx:        Optional[float] = None,
        search_cy:        Optional[float] = None,
        dist_sigma:       Optional[float] = None,
        lam_dist:         float           = 0.15,
        lam_cand_dir:     float           = 0.15,
    ) -> List[Tuple[np.ndarray, float]]:
        if not self._drm:
            box, score = self.match(frame, candidates, threshold=margin)
            if box is not None:
                return [(box, score)]
            return []

        if not candidates:
            return []

        ref_cx   = ref_bbox[0] + ref_bbox[2] / 2.0
        ref_cy   = ref_bbox[1] + ref_bbox[3] / 2.0
        vel_norm = float(np.linalg.norm(velocity)) + 1e-8

        scored: List[Tuple[np.ndarray, float]] = []

        for cand_bbox in candidates:
            cand_desc = _extract_descriptor(frame, cand_bbox)
            if cand_desc is None:
                continue

            anchor_scores = []
            for (dk_bbox, dk_desc, rho_k) in self._drm:
                s_iou = lam_iou * _iou(dk_bbox, cand_bbox)
                s_app = lam_app * _cos_sim(dk_desc, cand_desc)

                dk_cx      = dk_bbox[0] + dk_bbox[2] / 2.0
                dk_cy      = dk_bbox[1] + dk_bbox[3] / 2.0
                motion_vec = np.array([dk_cx - ref_cx, dk_cy - ref_cy])
                mot_norm   = float(np.linalg.norm(motion_vec)) + 1e-8
                pi_t       = float(np.dot(velocity, motion_vec) /
                                   (vel_norm * mot_norm))
                pi_t       = max(0.0, pi_t)
                s_mot      = lam_mot * pi_t

                age    = max(0, self._t - rho_k)
                s_time = lam_time * float(np.exp(-alpha * age))

                raw = s_iou + s_app + s_mot + s_time

                if distractor_bank:
                    pen = max(_cos_sim(dk_desc, nu) for nu in distractor_bank)
                    raw -= gamma * pen

                anchor_scores.append(raw)

            cand_score = max(anchor_scores) if anchor_scores else -np.inf

            if (search_cx is not None and search_cy is not None
                    and dist_sigma is not None and dist_sigma > 0):
                cand_cx    = cand_bbox[0] + cand_bbox[2] / 2.0
                cand_cy    = cand_bbox[1] + cand_bbox[3] / 2.0
                d          = np.hypot(cand_cx - search_cx, cand_cy - search_cy)
                cand_score -= lam_dist * (1.0 - np.exp(-0.5 * (d / dist_sigma) ** 2))

            # AFTER — centered on 0: wrong direction subtracts, right direction adds
            if lam_cand_dir > 0 and vel_norm > 1e-3:
                cand_cx  = cand_bbox[0] + cand_bbox[2] / 2.0
                cand_cy  = cand_bbox[1] + cand_bbox[3] / 2.0
                cand_vec = np.array([cand_cx - ref_cx, cand_cy - ref_cy])
                cand_d   = float(np.linalg.norm(cand_vec)) + 1e-8
                cos_dir  = float(np.dot(velocity, cand_vec) / (vel_norm * cand_d))
                cand_score += lam_cand_dir * cos_dir

            if cand_score > margin:
                scored.append((np.array(cand_bbox, dtype=int), float(cand_score)))

        if not scored:
            return []

        scored.sort(key=lambda x: x[1], reverse=True)

        if scored[0][1] >= skip_threshold:
            return [scored[0]]

        return scored[:top_k]

    def drm_size(self) -> int:
        return len(self._drm)

    def __len__(self):
        return len(self._buf)


# ─────────────────────────────────────────────────────────────────────────────
# EdgeDAMTracker
# ─────────────────────────────────────────────────────────────────────────────

class EdgeDAMTracker:
    """
    Occlusion-aware wrapper around SiamABCTracker with EKF motion estimation,
    background homography compensation, and DRM appearance memory.

    CHANGES vs previous version
    ───────────────────────────
    1. OCCLUSION ENTRY HYSTERESIS
       Score must stay below conf_threshold for `entry_patience` consecutive
       frames (default 3) before occlusion is declared.  The streak frames still
       return normal tracker output to the caller.  Memory admission is gated so
       low-confidence frames are never committed to RAM/DRM.  The streak length is
       folded into effective_skip at occlusion entry so the EKF rebuild ignores
       those corrupted frames.

    2. N-FRAME CANDIDATE COLLECTION WITH VELOCITY SCORING
       The old 2-step YOLO→verify is replaced by an N-frame collection pipeline
       (N = cand_collection_frames, default 3) followed by a single final DRM+verify
       phase that has access to measured candidate velocity:

         Phase 0              : SiamABC attempt (fast path – reuses reacq_threshold).
         Phases 1 … N        : YOLO detection only; store (bbox, descriptor) and
                               camera velocity for each frame.
         Phase N+1 (final)   : DRM match + velocity-augmented scoring + tracker
                               verification of top-k candidates.

       Velocity scoring is camera-compensated and lives in [0, 1]:
         - dir_score  = max(0, (cos(expected_vel, cand_vel) + 1) / 2)
           maps [-1,1] cosine to [0,1]; naturally penalises wrong-direction candidates
         - speed_score = clip(1 − |speed_ratio − 1|, 0, 1)
           is 1 when speeds match, falls off linearly, 0 when 2× off or stationary
         - vel_score   = 0.6 · dir_score + 0.4 · speed_score  ∈ [0, 1]
         - contribution to DRM score: lam_cand_vel · vel_score

       Edge-case handling:
         - out_of_frame → lam_cand_vel forced to 0 (velocity is meaningless at edge)
         - long_distance / tiny object → lam_cand_vel halved
         - expected speed < vel_score_min_speed → vel_score = 0.5 (neutral, no bias)
         - candidate tracked across < 2 frames → vel_score = 0.5 (neutral)

    3. EKF-UNCERTAINTY-SCALED dist_sigma
       dist_sigma for the DRM spatial penalty is now:
           max(drm_dist_sigma_factor · max(obj_w, obj_h),  ekf_uncertainty · 1.5)
       When the EKF is very uncertain (large ROI), the Gaussian distance penalty
       weakens automatically so a correct candidate far from the stale predicted
       position is not unfairly penalised.

    4. TINY-OBJECT ROI PARAMETER SET
       When long_distance_mode is active or the object is detected as tiny,
       a separate set of ROI-expansion parameters is used:
       tiny_roi_start_expand, tiny_yolo_search_expand,
       tiny_search_expand_growth_factor, tiny_search_expand_growth_every,
       tiny_search_expand_max.
    """

    def __init__(
        self,
        siam_tracker:                 object,           # ORTrack
        yolo_weights:               str   = "yolo11n.pt",
        conf_threshold:             float = 0.60,
        reacq_threshold:            float = 0.55,
        yolo_conf:                  float = 0.30,
        yolo_iou:                   float = 0.45,
        app_match_threshold:        float = 0.72,
        nudge_alpha:                float = 0.30,
        tau_occ:                    float = 0.40,
        beta:                       float = 0.06,
        mem_capacity:               int   = 20,
        tau_iou:                    float = 0.40,
        tau_area:                   float = 0.25,
        ncc_threshold:              float = 0.70,
        ncc_expand:                 float = 2.5,
        conf_history_len:           int   = 200,
        history_decay:              float = 0.5,
        history_skip_last:          int   = 2,
        # ── ROI params ────────────────────────────────────────────────────
        yolo_search_expand:         float = 5.0,
        roi_start_expand:           float = 1.5,
        size_history_len:           int   = 40,
        # ── DRM params ────────────────────────────────────────────────────
        drm_capacity:               int   = 8,
        drm_tau_sim:                float = 0.85,
        drm_window_W:               int   = 10,
        drm_mmin:                   int   = 3,
        drm_lam_iou:                float = 0.40,
        drm_lam_app:                float = 0.30,
        drm_lam_mot:                float = 0.20,
        drm_lam_time:               float = 0.10,
        drm_alpha:                  float = 0.05,
        drm_gamma:                  float = 0.30,
        drm_margin:                 float = 0.35,
        drm_top_k:                  int   = 3,
        drm_skip_threshold:         float = 0.80,
        drm_lam_dist:               float = 0.15,
        drm_dist_sigma_factor:      float = 2.5,
        drm_lam_cand_dir:           float = 0.15,
        # ── NEW: velocity scoring in final DRM phase ──────────────────────
        drm_lam_cand_vel:           float = 0.20,
        vel_score_min_speed:        float = 0.5,
        # ── EKF params ────────────────────────────────────────────────────
        ekf_process_noise:          float = 2.0,
        ekf_meas_noise:             float = 5.0,
        homo_max_corners:           int   = 200,
        homo_inlier_threshold:      float = 0.50,
        # ── Shrinkage detection params ────────────────────────────────────
        shrinkage_min_drop_frac:    float = 0.06,
        shrinkage_max_lookback:     int   = 60,
        # ── Velocity estimation (normal tracking) ─────────────────────────
        velocity_lookback:          int   = 3,
        velocity_smooth_alpha:      float = 0.4,
        # ── Memory leak fix ───────────────────────────────────────────────
        distractor_bank_maxlen:     int   = 50,
        # ── Misc ──────────────────────────────────────────────────────────
        velocity_decay:             float = 0.95,
        search_expand_growth_factor: float = 1.2,
        search_expand_growth_every:  int   = 5,
        search_expand_max:          float = 15.0,
        long_distance_conf_threshold: float = 0.35,
        long_distance_area_fraction:  float = 0.004,
        long_distance_mode:           bool  = False,
        enter_occlusion_on_loss:      bool  = True,
        velocity_window_average:      int   = 80,
        occlusion_patience:           int   = 5,
        occlusion_hysteresis:         float = 0.10,
        # ── NEW: Tiny/long-distance object ROI parameters ─────────────────
        tiny_roi_start_expand:            float = 3.0,
        tiny_yolo_search_expand:          float = 20.0,
        tiny_search_expand_growth_factor: float = 1.3,
        tiny_search_expand_growth_every:  int   = 5,
        tiny_search_expand_max:           float = 40.0,
        # ── NEW: Occlusion entry hysteresis ───────────────────────────────
        entry_patience:             int   = 3,
        # ── NEW: Multi-frame candidate collection ─────────────────────────
        cand_collection_frames:     int   = 3,

        vel_dir_hard_gate:          float = 0.5,   # |cos| threshold below which score → 0.05
        yolo_filter_class:          bool  = False, # filter candidates to target class
        yolo_class_detect_frames:   int   = 5,     # stride frames to detect class at init

    ):
        self.tracker              = siam_tracker
        self.yolo                 = YOLO(yolo_weights)
        self.conf_threshold       = conf_threshold
        self.reacq_threshold      = reacq_threshold
        self.yolo_conf            = yolo_conf
        self.yolo_iou_thr         = yolo_iou
        self.app_match_threshold  = app_match_threshold
        self.nudge_alpha          = nudge_alpha
        self.tau_occ              = tau_occ
        self.beta                 = beta
        self.ncc_threshold        = ncc_threshold
        self.ncc_expand           = ncc_expand
        self.conf_history_len     = conf_history_len
        self.history_decay        = history_decay
        self.history_skip_last    = history_skip_last
        self.yolo_search_expand   = yolo_search_expand
        self.roi_start_expand     = roi_start_expand
        self.size_history_len     = size_history_len

        # EKF
        self.ekf_process_noise     = ekf_process_noise
        self.ekf_meas_noise        = ekf_meas_noise
        self.homo_max_corners      = homo_max_corners
        self.homo_inlier_threshold = homo_inlier_threshold

        # Shrinkage detection
        self.shrinkage_min_drop_frac = shrinkage_min_drop_frac
        self.shrinkage_max_lookback  = shrinkage_max_lookback

        # Velocity (finite-diff, normal tracking)
        self.velocity_lookback      = velocity_lookback
        self.velocity_smooth_alpha  = velocity_smooth_alpha

        # Memory leak fix
        self._distractor_bank_maxlen = distractor_bank_maxlen
        self.velocity_window_average = velocity_window_average

        self.search_expand_growth_factor = search_expand_growth_factor
        self.search_expand_growth_every  = search_expand_growth_every
        self.search_expand_max           = search_expand_max
        self._occ_frames: int            = 0

        self._drm_dist_sigma_factor = drm_dist_sigma_factor
        self._size_history: deque   = deque(maxlen=size_history_len)
        self._cam_disp_history: deque = deque(maxlen=conf_history_len)
        self._vel_history: deque = deque(maxlen=200)

        self.long_distance_conf_threshold  = long_distance_conf_threshold
        self.long_distance_area_fraction   = long_distance_area_fraction
        self.long_distance_mode            = long_distance_mode
        self.recovered_early_occlusion     = True
        self.enter_occlusion_on_loss       = enter_occlusion_on_loss
        self._drm_kwargs = dict(
            lam_iou        = drm_lam_iou,
            lam_app        = drm_lam_app,
            lam_mot        = drm_lam_mot,
            lam_time       = drm_lam_time,
            alpha          = drm_alpha,
            gamma          = drm_gamma,
            margin         = drm_margin,
            top_k          = drm_top_k,
            skip_threshold = drm_skip_threshold,
            lam_dist       = drm_lam_dist,
            lam_cand_dir   = drm_lam_cand_dir,
        )

        # ── NEW params ────────────────────────────────────────────────────
        self._drm_lam_cand_dir = drm_lam_cand_dir
        self._vel_score_min_speed     = vel_score_min_speed
        self._entry_patience          = max(1, entry_patience)
        self._cand_collection_frames  = max(1, cand_collection_frames)
        # Tiny-object ROI
        self.tiny_roi_start_expand            = tiny_roi_start_expand
        self.tiny_yolo_search_expand          = tiny_yolo_search_expand
        self.tiny_search_expand_growth_factor = tiny_search_expand_growth_factor
        self.tiny_search_expand_growth_every  = tiny_search_expand_growth_every
        self.tiny_search_expand_max           = tiny_search_expand_max

        self.memory = AppearanceMemory(
            capacity     = mem_capacity,
            tau_iou      = tau_iou,
            tau_area     = tau_area,
            drm_capacity = drm_capacity,
            tau_sim      = drm_tau_sim,
            window_W     = drm_window_W,
            mmin         = drm_mmin,
        )

        # ── Runtime state ─────────────────────────────────────────────────
        self.current_bbox:       Optional[np.ndarray] = None
        self.held_box:           Optional[np.ndarray] = None
        self.in_occlusion:       bool                 = False
        self.frame_idx:          int                  = 0
        self.velocity:           np.ndarray           = np.zeros(2)
        self.prev_gray:          Optional[np.ndarray] = None
        self.init_frame:         Optional[np.ndarray] = None
        self.init_bbox:          Optional[np.ndarray] = None
        self._last_yolo:         List                 = []
        self._yolo_cache:        List                 = []
        self._distractor_bank:   deque                = deque(maxlen=distractor_bank_maxlen)
        self._out_of_frame:      bool                 = False
        self._exit_edge:         Optional[str]        = None
        self._search_cx:         Optional[float]      = None
        self._search_cy:         Optional[float]      = None
        self._conf_history:      deque                = deque(maxlen=conf_history_len)
        self._center_history:    deque                = deque(maxlen=200)
        self._cam_vel_history:   deque                = deque(maxlen=200)


        self._vel_dir_hard_gate       = vel_dir_hard_gate
        self._yolo_filter_class       = yolo_filter_class
        self._yolo_class_detect_frames = yolo_class_detect_frames
        self._target_class_id:  Optional[int] = None   # set during warm-up
        self._class_warmup_done: bool          = False

        # ── Phase state ───────────────────────────────────────────────────
        # Phase 0         : SiamABC attempt
        # Phases 1 … N   : YOLO candidate collection (N = cand_collection_frames)
        # Phase N+1       : Final DRM + velocity scoring + verify
        self._occ_phase:          int  = 0
        self._pending_candidates: List = []   # kept for compat but unused

        # ── NEW: multi-frame collection buffers ───────────────────────────
        # _cand_frames[k]   = list of (bbox, desc) at collection frame k
        # _occ_cam_vels[k]  = camera velocity vector AT collection frame k
        #                     (motion from frame k-1 to frame k)
        # Both grow together (same index → same frame).
        self._cand_frames:   List = []
        self._occ_cam_vels:  List = []

        # ── NEW: entry hysteresis ─────────────────────────────────────────
        # Counts consecutive frames where tracker score < conf_threshold.
        # Occlusion is declared only when streak >= entry_patience.
        self._entry_streak: int = 0

        self.ekf: Optional[BBoxEKF] = None
        self._last_H:          Optional[np.ndarray] = None
        self._last_H_reliable: bool                 = False

        self._flow_scale   = 0.5
        self._cached_pts   = None
        self._cached_shape = None

        self._low_score_streak = 0
        self._occlusion_patience   = occlusion_patience
        self._occlusion_hysteresis = occlusion_hysteresis
        self._gated_score = 1.0

    # ── Public API ────────────────────────────────────────────────────────────

    def initialize(self, frame: np.ndarray, bbox) -> None:
        self.tracker.enable_tta()
        bbox = np.array(bbox, dtype=int)
        self.tracker.initialize(frame, bbox)
        self.current_bbox     = bbox.copy()
        self.held_box         = bbox.copy()
        self.in_occlusion     = False
        self.frame_idx        = 0
        self.velocity         = np.zeros(2)
        self.prev_gray        = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        self.init_frame       = frame.copy()
        self.init_bbox        = bbox.copy()
        self._distractor_bank = deque(maxlen=self._distractor_bank_maxlen)
        self._out_of_frame    = False
        self._exit_edge       = None
        self._search_cx       = None
        self._search_cy       = None
        self._size_history.clear()
        self.memory.reset()
        self._conf_history.clear()
        self._cam_disp_history.clear()
        self._yolo_cache      = []
        self._occ_frames      = 0
        self._occ_phase       = 0
        self._pending_candidates = []
        self.recovered_early_occlusion = True
        self._last_H          = None
        self._last_H_reliable = False
        self._last_yolo       = []

        # NEW resets
        self._entry_streak  = 0
        self._cand_frames   = []
        self._occ_cam_vels  = []

        self._target_class_id  = None
        self._class_warmup_done = False

        self.ekf = BBoxEKF(bbox,
                           process_noise=self.ekf_process_noise,
                           meas_noise=self.ekf_meas_noise)

        desc = _extract_descriptor(frame, bbox)

        self._vel_history.clear()
        self._center_history.clear()
        self._cam_vel_history.clear()
        if desc is not None:
            self.memory.try_admit(bbox, desc, bbox)

    def update(self, frame: np.ndarray) -> Tuple[np.ndarray, float, bool, List]:
        self.frame_idx  += 1
        self._last_yolo  = []
        self._yolo_cache = []

        H, H_reliable, current_gray = self._estimate_homography(frame)
        self._last_H          = H
        self._last_H_reliable = H_reliable

        if self.in_occlusion and self._out_of_frame:
            self.ekf.P = self.ekf.P + self.ekf.Q
        else:
            self.ekf.predict(H=H, H_reliable=H_reliable)

        if self.in_occlusion:
            bbox, score = self._occlusion_update(frame)
        else:
            bbox, score = self._normal_update(frame)

        self.prev_gray = current_gray

        if self.in_occlusion:
            return np.zeros(4, dtype=int), 0.0, True, self._last_yolo

        return bbox.copy(), float(score), self.in_occlusion, self._last_yolo

    # ── Normal path ───────────────────────────────────────────────────────────

    def _normal_update(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        pred_bbox, score, _ = self.tracker.update(frame)
        pred_bbox = np.array(pred_bbox, dtype=int)


        if (self._yolo_filter_class
        and not self._class_warmup_done
        and self.frame_idx <= self._yolo_class_detect_frames * 3
        and self.frame_idx % max(1, self._yolo_class_detect_frames) == 0):
            self._try_detect_target_class(frame)
            if self.frame_idx >= self._yolo_class_detect_frames * 2:
                self._maybe_commit_target_class()

        # ── Threshold (long-distance mode) ───────────────────────────────
        if self.frame_idx > 0:
            long_distanced_object = self._is_long_distance(frame)
            effective_threshold = (
                self.long_distance_conf_threshold
                if long_distanced_object
                else self.conf_threshold
            )
            self.tracker.offset = (
                self.tracker.tracking_config["search_context"]
                if not long_distanced_object
                else self.tracker.tracking_config["search_context"] + 0.5
            )
        else:
            effective_threshold = 0.0

        # ── CHANGED: Entry hysteresis ─────────────────────────────────────
        # Accumulate consecutive low-score frames; only enter occlusion after
        # entry_patience such frames in a row.
        if (score < effective_threshold
                and self.frame_idx >= 30
                and self.enter_occlusion_on_loss):
            self._entry_streak += 1
        else:
            self._entry_streak = 0

        if (self._entry_streak >= self._entry_patience
                and self.frame_idx >= 30
                and self.enter_occlusion_on_loss):

            entry_streak_val   = self._entry_streak
            self._entry_streak = 0
            self.in_occlusion  = True

            # Out-of-frame detection
            is_exiting, exit_edge = self._detect_exit_direction(frame)
            self._out_of_frame = is_exiting
            self._exit_edge    = exit_edge

            loss_cause = self._classify_loss_cause()
            if is_exiting:
                loss_cause = 'out_of_frame'

            print(
                f"[occlusion entry] frame={self.frame_idx}  "
                f"loss_cause={loss_cause}  "
                f"out_of_frame={self._out_of_frame}  exit_edge={self._exit_edge}  "
                f"entry_streak={entry_streak_val}"
            )

            area_skip    = self._detect_shrinkage_onset(
                max_lookback  = self.shrinkage_max_lookback,
                min_drop_frac = self.shrinkage_min_drop_frac,
            )
            drift_skip   = self._detect_center_drift_skip(
                max_lookback=self.shrinkage_max_lookback)
            dynamic_skip = max(area_skip, drift_skip)

            # For camera-motion and out-of-frame losses shrinkage/drift skip
            # is irrelevant, but we always skip at least the streak frames
            # because those bboxes had low-confidence tracker output.
            if loss_cause in ('camera_motion', 'out_of_frame'):
                effective_skip = 0
            else:
                effective_skip = max(dynamic_skip, entry_streak_val)

            print(
                f"[occlusion entry] frame={self.frame_idx}  "
                f"loss_cause={loss_cause}  dynamic_skip={dynamic_skip}  "
                f"entry_streak_skip={entry_streak_val}  "
                f"effective_skip={effective_skip}  "
                f"history_len={len(self._conf_history)}"
            )

            # Reset all phase / collection state at every occlusion entry
            self._occ_phase          = 0
            self._pending_candidates = []
            self._cand_frames        = []   # NEW
            self._occ_cam_vels       = []   # NEW

            self.ekf = self._rebuild_ekf_from_clean_history(
                skip_override=effective_skip)
            self.ekf.predict(H=self._last_H,
                             H_reliable=self._last_H_reliable)
            self._init_search_centre_from_history(
                skip_override=effective_skip)
            self.tracker.dynamic_update = False
            self._occ_frames = 0
            self.tracker.disable_tta()
            return self._occlusion_update(frame)

        # ── Motion model DISABLED as output — use SiamABC bbox directly ──
        self.ekf.update(pred_bbox)

        cam_disp = self._h_translation_magnitude(self._last_H, frame)
        self._cam_disp_history.append(cam_disp)

        h_fr, w_fr = frame.shape[:2]
        if self._last_H is not None:
            cx, cy = w_fr / 2.0, h_fr / 2.0
            denom  = (self._last_H[2,0]*cx + self._last_H[2,1]*cy
                      + self._last_H[2,2] + 1e-8)
            ncx    = ((self._last_H[0,0]*cx + self._last_H[0,1]*cy
                       + self._last_H[0,2]) / denom)
            ncy    = ((self._last_H[1,0]*cx + self._last_H[1,1]*cy
                       + self._last_H[1,2]) / denom)
            self._cam_vel_history.append(np.array([ncx - cx, ncy - cy]))
        else:
            self._cam_vel_history.append(np.zeros(2))

        cx = float(pred_bbox[0] + pred_bbox[2] / 2.0)
        cy = float(pred_bbox[1] + pred_bbox[3] / 2.0)
        self._center_history.append(np.array([cx, cy]))

        self.velocity = self._compute_velocity_from_history(pred_bbox)
        self._vel_history.append(self.velocity.copy())

        # CHANGED: only admit to memory when tracker score is good
        if score >= effective_threshold:
            desc = _extract_descriptor(frame, pred_bbox)
            if desc is not None:
                self.memory.try_admit(pred_bbox, desc, self.current_bbox)

        self.current_bbox = pred_bbox.copy()
        self.held_box     = pred_bbox.copy()
        self._size_history.append((int(pred_bbox[2]), int(pred_bbox[3])))
        self._conf_history.append((pred_bbox.copy(), self.velocity.copy(),
                                   self._last_H, self._last_H_reliable))

        return pred_bbox, score

    # ── Velocity (finite-difference, used during normal tracking) ─────────────

    def _compute_velocity_from_history(
        self,
        current_bbox:  np.ndarray,
        lookback:      Optional[int]   = None,
        smooth_alpha:  Optional[float] = None,
    ) -> np.ndarray:
        lb    = lookback     if lookback     is not None else self.velocity_lookback
        alpha = smooth_alpha if smooth_alpha is not None else self.velocity_smooth_alpha

        if not self._conf_history:
            return np.zeros(2)

        history = list(self._conf_history)
        n_back  = min(lb, len(history))
        if n_back == 0:
            return np.zeros(2)

        curr_cx = current_bbox[0] + current_bbox[2] / 2.0
        curr_cy = current_bbox[1] + current_bbox[3] / 2.0

        ref_bbox = history[-n_back][0]
        ref_cx   = ref_bbox[0] + ref_bbox[2] / 2.0
        ref_cy   = ref_bbox[1] + ref_bbox[3] / 2.0

        raw_vx = (curr_cx - ref_cx) / n_back
        raw_vy = (curr_cy - ref_cy) / n_back

        vx = alpha * raw_vx + (1.0 - alpha) * self.velocity[0]
        vy = alpha * raw_vy + (1.0 - alpha) * self.velocity[1]
        return np.array([vx, vy])

    # ── Search-centre init at occlusion entry ─────────────────────────────────

    def _init_search_centre_from_history(self, skip_override=None) -> None:
        history = list(self._conf_history)
        skip    = (min(self.history_skip_last, len(history) - 1)
                   if skip_override is None else skip_override)
        clean   = history[:len(history) - skip] if skip > 0 else history

        if clean:
            last_clean_bbox  = clean[-1][0].astype(float)
            self._search_cx  = last_clean_bbox[0] + last_clean_bbox[2] / 2.0
            self._search_cy  = last_clean_bbox[1] + last_clean_bbox[3] / 2.0
        else:
            self._search_cx  = float(self.current_bbox[0]
                                     + self.current_bbox[2] / 2.0)
            self._search_cy  = float(self.current_bbox[1]
                                     + self.current_bbox[3] / 2.0)

    # ── Occlusion update — dispatcher ─────────────────────────────────────────

    def _occlusion_update(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        h_fr, w_fr = frame.shape[:2]

        ekf_raw         = self.ekf.get_bbox()
        self._search_cx = float(ekf_raw[0] + ekf_raw[2] / 2.0)
        self._search_cy = float(ekf_raw[1] + ekf_raw[3] / 2.0)
        self.held_box   = self._clamp_bbox_to_frame(ekf_raw, frame)
        self.velocity   = self.ekf.get_velocity()
        self._occ_frames += 1

        # Pin search centre to exit edge when out-of-frame
        if self._out_of_frame and self._exit_edge is not None:
            if self._exit_edge == 'right':
                self._search_cx = float(w_fr - 1)
            elif self._exit_edge == 'left':
                self._search_cx = 0.0
            elif self._exit_edge == 'bottom':
                self._search_cy = float(h_fr - 1)
            elif self._exit_edge == 'top':
                self._search_cy = 0.0

        # Out-of-frame state management
        if self._out_of_frame:
            ekf_inside = (0 <= self._search_cx < w_fr
                          and 0 <= self._search_cy < h_fr)
            vel_inward = False
            if ekf_inside and self._exit_edge is not None:
                vel_inward = {
                    'right':  float(self.velocity[0]) < 0,
                    'left':   float(self.velocity[0]) > 0,
                    'bottom': float(self.velocity[1]) < 0,
                    'top':    float(self.velocity[1]) > 0,
                }.get(self._exit_edge, True)
            if ekf_inside and vel_inward:
                self._out_of_frame = False
                self._exit_edge    = None
        else:
            if (self._search_cx < 0 or self._search_cx >= w_fr or
                    self._search_cy < 0 or self._search_cy >= h_fr):
                if   self._search_cx >= w_fr: self._exit_edge = 'right'
                elif self._search_cx < 0:     self._exit_edge = 'left'
                elif self._search_cy >= h_fr: self._exit_edge = 'bottom'
                else:                         self._exit_edge = 'top'
                self._out_of_frame = True

        # CHANGED: 3-state dispatcher
        # Phase 0              → _occ_phase_siam
        # Phases 1 … N        → _occ_phase_collect  (candidate collection)
        # Phase N+1            → _occ_phase_final_drm (velocity-scored DRM + verify)
        if self._occ_phase == 0:
            return self._occ_phase_siam(frame)
        elif 1 <= self._occ_phase <= self._cand_collection_frames:
            return self._occ_phase_collect(frame)
        else:
            return self._occ_phase_final_drm(frame)

    # ── Phase 0: SiamABC attempt ──────────────────────────────────────────────

    def _occ_phase_siam(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        rx, ry, rw, rh = self._get_yolo_search_roi(frame=frame)

        obj_w, obj_h = self._get_median_size()

        roi_cx    = rx + rw / 2.0
        roi_cy    = ry + rh / 2.0
        seed_bbox = np.array([
            int(roi_cx - obj_w / 2.0),
            int(roi_cy - obj_h / 2.0),
            obj_w, obj_h,
        ], dtype=int)
        seed_bbox = self._clamp_bbox_to_frame(seed_bbox, frame)

        self.tracker.tracking_state.bbox = seed_bbox
        pred_bbox, score, _ = self.tracker.update(frame)
        pred_bbox = np.array(pred_bbox, dtype=int)

        if score >= self.reacq_threshold:
            pred_desc = _extract_descriptor(frame, pred_bbox)

            # Compute single-frame candidate velocity: displacement from last clean position
            cand_vel_phase0 = None
            if self._conf_history:
                last_bbox = self._conf_history[-1][0]
                dx = (pred_bbox[0] + pred_bbox[2]/2.0) - (last_bbox[0] + last_bbox[2]/2.0)
                dy = (pred_bbox[1] + pred_bbox[3]/2.0) - (last_bbox[1] + last_bbox[3]/2.0)
                # Camera-compensate
                cam = self._cam_vel_from_H(frame)
                cand_vel_phase0 = np.array([dx - cam[0], dy - cam[1]])

            drm_results = self.memory.drm_match(
            frame           = frame,
            candidates      = [pred_bbox],
            ref_bbox        = self.held_box,
            velocity        = self.velocity,
            distractor_bank = self._distractor_bank,
            search_cx       = self._search_cx,
            search_cy       = self._search_cy,
            dist_sigma      = self._effective_dist_sigma(frame),
            **self._drm_kwargs,
            )  # lam_cand_dir no longer in kwargs

            drm_score = drm_results[0][1] if drm_results else -1.0

            # Apply the same direction scoring the final phase uses
            lam_dir = self._drm_lam_cand_dir
            if lam_dir > 0 and cand_vel_phase0 is not None:
                dir_score = self._compute_velocity_score(cand_vel_phase0, self.velocity)
                drm_score += lam_dir * (2.0 * dir_score - 1.0)

            drm_ok = drm_score >= self.app_match_threshold

            print(f"[occ frame {self._occ_frames}] phase=siam  "
                  f"score={score:.3f}  drm={drm_score:.3f}  pass={drm_ok}")

            if drm_ok:
                self.recovered_early_occlusion = True
                return self._commit_reacquisition(
                    frame, pred_bbox, pred_desc, score)

            self.tracker.tracking_state.bbox = self.held_box.copy()

        # Advance to candidate collection phase
        self._cand_frames  = []
        self._occ_cam_vels = []
        self._occ_phase    = 1
        return self.held_box, score

    # ── Phases 1…N: Candidate collection ─────────────────────────────────────

    def _occ_phase_collect(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Run YOLO, extract descriptors, and store candidates + camera velocity
        for this frame.  No matching happens here — it all happens in the final
        phase so every candidate has a full multi-frame trajectory.
        """
        # Camera velocity for THIS frame (motion from prev frame → this frame)
        cam_vel = self._cam_vel_from_H(frame)
        self._occ_cam_vels.append(cam_vel)

        # YOLO detection
        detections      = self._yolo_detect(frame)
        self._last_yolo = detections

        # Extract descriptors; store (bbox, desc) pairs
        frame_cands = []
        for bbox in detections:
            desc = _extract_descriptor(frame, bbox)
            if desc is not None:
                frame_cands.append((np.array(bbox, dtype=int), desc.copy()))

        self._cand_frames.append(frame_cands)

        # Update distractor bank
        for det in detections:
            if _iou(det, self.held_box) >= self.tau_occ:
                det_desc = _extract_descriptor(frame, det)
                if det_desc is not None:
                    self._distractor_bank.append(det_desc)

        collection_phase_num = self._occ_phase   # 1-indexed within collection
        print(
            f"[occ frame {self._occ_frames}] "
            f"phase=collect({collection_phase_num}/{self._cand_collection_frames})  "
            f"detections={len(detections)}  stored={len(frame_cands)}"
        )

        # Advance: if we just finished the last collection frame go to final;
        # otherwise stay in collection.
        self._occ_phase += 1
        # (dispatcher will route to _occ_phase_final_drm when phase > N)

        return self.held_box, 0.0

    # ── Phase N+1: Velocity-scored DRM + verification ─────────────────────────

    def _occ_phase_final_drm(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Final occlusion recovery phase.

        1. Pull candidates from the last non-empty collection frame.
        2. Build camera-compensated velocity vectors for each candidate by
        tracing it back through ALL earlier collection frames.
        Candidates missing from ANY prior frame are excluded from DRM.
        3. Run DRM matching on fully-tracked candidates only.
        4. Augment DRM scores with a velocity consistency term (lam_cand_vel).
        5. Verify top-k candidates with the tracker (1-frame motion compensation).
        6. On success → commit reacquisition.  On full failure → reset to phase 0.
        """
        def _reset():
            self._occ_phase    = 0
            self._cand_frames  = []
            self._occ_cam_vels = []
            self.tracker.tracking_state.bbox = self.held_box.copy()

        # ── Find last non-empty collection frame ──────────────────────────────
        last_idx = -1
        for i in range(len(self._cand_frames) - 1, -1, -1):
            if self._cand_frames[i]:
                last_idx = i
                break

        if last_idx == -1:
            print(f"[occ frame {self._occ_frames}] phase=final_drm  "
                f"no candidates in any collection frame — resetting")
            _reset()
            return self.held_box, 0.0

        last_frame_cands = self._cand_frames[last_idx]
        last_cand_bboxes = [b for (b, _) in last_frame_cands]

        # ── Build camera-compensated velocities for candidates ────────────────
        # Returns None for any candidate not found in ALL prior collection frames.
        cand_vels = self._build_candidate_velocities(last_idx)

        # ── Filter: only keep candidates that appeared in every collection frame -
        # Single collection frame is a special case: no prior frames exist so
        # all candidates pass through (vel=None → neutral score).
        single_frame_mode = (last_idx == 0)

        fully_tracked_bboxes = [
            bbox
            for bbox, vel in zip(last_cand_bboxes, cand_vels)
            if vel is not None or single_frame_mode
        ]

        # We still use ALL candidates for nudging so held_box doesn't freeze
        # even when no candidate survived the tracking filter.
        print(
            f"[occ frame {self._occ_frames}] phase=final_drm  "
            f"last_cands={len(last_cand_bboxes)}  "
            f"fully_tracked={len(fully_tracked_bboxes)}  "
            f"drm_size={self.memory.drm_size()}  ram={len(self.memory)}  "
            f"ekf_unc={self.ekf.get_uncertainty():.1f}px"
        )

        if not fully_tracked_bboxes:
            print(f"[occ frame {self._occ_frames}] phase=final_drm  "
                f"no fully-tracked candidates — resetting")
            if last_cand_bboxes:
                self.held_box = self._nudge_toward_nearest(frame, last_cand_bboxes)
                self.ekf.nudge_position(self.held_box)
            _reset()
            return self.held_box, 0.0

        # ── DRM matching with EKF-uncertainty-aware dist_sigma ────────────────
        dist_sigma = self._effective_dist_sigma(frame)

        drm_results = self.memory.drm_match(
            frame           = frame,
            candidates      = fully_tracked_bboxes,   # only fully-tracked candidates
            ref_bbox        = self.held_box,
            velocity        = self.velocity,
            distractor_bank = self._distractor_bank,
            search_cx       = self._search_cx,
            search_cy       = self._search_cy,
            dist_sigma      = dist_sigma,
            **self._drm_kwargs,
        )

        if not drm_results:
            if last_cand_bboxes:
                self.held_box = self._nudge_toward_nearest(frame, last_cand_bboxes)
                self.ekf.nudge_position(self.held_box)
            _reset()
            return self.held_box, 0.0

        # ── Determine effective velocity weight ───────────────────────────────
        lam_dir  = self._drm_lam_cand_dir  
        if self._out_of_frame:
            lam_dir = 0.0
        elif self._is_long_distance(frame):
            lam_dir *= 0.5

        expected_vel = self.velocity  # camera-compensated EKF velocity

        # ── Match DRM result bboxes back to candidate indices ────────────────
        # DRM may return slightly clipped copies of bboxes, so match by IoU.
        # We match against last_cand_bboxes (full list) because cand_vels is
        # indexed against it.
        def _find_cand_idx(drm_bbox):
            best_iou, best_idx = 0.3, None
            for i, cb in enumerate(last_cand_bboxes):
                v = _iou(drm_bbox, cb)
                if v > best_iou:
                    best_iou, best_idx = v, i
            return best_idx

        # ── Augment DRM scores with velocity term ─────────────────────────────
        # Every candidate in drm_results is guaranteed to be fully-tracked
        # (None was already excluded above), so vel should never be None here.
        # We guard anyway for safety.
        final_scored = []
        for (drm_bbox, drm_score) in drm_results:
            cand_idx  = _find_cand_idx(drm_bbox)
            vel       = (cand_vels[cand_idx]
                        if cand_idx is not None and cand_idx < len(cand_vels)
                        else None)
            # _compute_velocity_score compares candidate's ACTUAL measured velocity
            # (camera-compensated, from _build_candidate_velocities) to EKF velocity.
            # This is the correct implementation of "does this candidate move like the target?"
            dir_score = (self._compute_velocity_score(vel, expected_vel)
                        if vel is not None else 0.5)

            augmented = drm_score + lam_dir * (2.0 * dir_score - 1.0)
            final_scored.append((drm_bbox, augmented, dir_score))

        final_scored.sort(key=lambda x: x[1], reverse=True)

        # ── Verify top-k with tracker ─────────────────────────────────────────
        # Candidates are from last_idx collection frame (1+ frames ago).
        # Compensate by 1 frame of EKF velocity before verifying.
        vx    = float(self.velocity[0])
        vy    = float(self.velocity[1])
        top_k = self._drm_kwargs.get('top_k', 3)

        for (match_bbox, match_score, vel_score) in final_scored[:top_k]:
            adjusted    = match_bbox.astype(float).copy()
            adjusted[0] += vx
            adjusted[1] += vy
            adjusted    = self._clamp_bbox_to_frame(
                np.array(adjusted, dtype=int), frame)

            self.tracker.dynamic_update = False
            verify_bbox, verify_score, _ = self.tracker.run_track_for_candidate(
                frame, adjusted)
            verify_bbox = np.array(verify_bbox, dtype=int)

            print(
                f"[occ frame {self._occ_frames}] phase=final_drm_verify  "
                f"drm={match_score:.3f}  vel={vel_score:.3f}  "
                f"verify={verify_score:.3f}  "
                f"pass={verify_score >= self.reacq_threshold}"
            )

            if verify_score >= self.reacq_threshold:
                self.recovered_early_occlusion = True
                desc = _extract_descriptor(frame, verify_bbox)
                self._cand_frames  = []
                self._occ_cam_vels = []
                return self._commit_reacquisition(
                    frame, verify_bbox, desc, verify_score)

        # All verification failed — back to SiamABC next frame
        _reset()
        return self.held_box, 0.0

    # ── Shared reacquisition exit ─────────────────────────────────────────────

    def _commit_reacquisition(
        self,
        frame: np.ndarray,
        bbox:  np.ndarray,
        desc:  Optional[np.ndarray],
        score: float,
    ) -> Tuple[np.ndarray, float]:
        self.ekf.update(bbox)
        ekf_bbox      = self.ekf.get_bbox()
        self.velocity = self.ekf.get_velocity()

        self.in_occlusion  = False
        self._out_of_frame = False
        self._exit_edge    = None
        self._occ_frames   = 0
        self._occ_phase    = 0
        self._pending_candidates = []
        self._cand_frames  = []
        self._occ_cam_vels = []
        self._entry_streak = 0   # ensure clean state on return to normal tracking
        self.tracker.enable_tta()
        self.tracker.dynamic_update = self.tracker.tracking_config["dynamic_update"]

        if desc is not None:
            self.memory.try_admit(ekf_bbox, desc, self.held_box)
        self.current_bbox = ekf_bbox.copy()
        self.held_box     = ekf_bbox.copy()
        self.tracker.tracking_state.bbox = ekf_bbox.copy()
        self._search_cx   = float(ekf_bbox[0] + ekf_bbox[2] / 2.0)
        self._search_cy   = float(ekf_bbox[1] + ekf_bbox[3] / 2.0)

        cx = float(ekf_bbox[0] + ekf_bbox[2] / 2.0)
        cy = float(ekf_bbox[1] + ekf_bbox[3] / 2.0)
        self._center_history.append(np.array([cx, cy]))

        cam_disp = np.zeros(2)
        if self._last_H is not None:
            h_fr, w_fr = frame.shape[:2]
            cx, cy = w_fr / 2.0, h_fr / 2.0
            denom  = (self._last_H[2,0]*cx + self._last_H[2,1]*cy
                      + self._last_H[2,2] + 1e-8)
            ncx    = ((self._last_H[0,0]*cx + self._last_H[0,1]*cy
                       + self._last_H[0,2]) / denom)
            ncy    = ((self._last_H[1,0]*cx + self._last_H[1,1]*cy
                       + self._last_H[1,2]) / denom)
            cam_disp = np.array([ncx - cx, ncy - cy])
        self._cam_vel_history.append(cam_disp)

        self._conf_history.append((
            ekf_bbox.copy(),
            self.velocity.copy(),
            self._last_H,
            self._last_H_reliable,
        ))
        return ekf_bbox, score

    # ─────────────────────────────────────────────────────────────────────────
    # NEW helpers
    # ─────────────────────────────────────────────────────────────────────────

    def _get_median_size(self) -> Tuple[int, int]:
        """Return median (width, height) of the tracked object in pixels."""
        if self._size_history:
            w = max(1, int(np.median([s[0] for s in self._size_history])))
            h = max(1, int(np.median([s[1] for s in self._size_history])))
        else:
            w = max(1, int(self.held_box[2]))
            h = max(1, int(self.held_box[3]))
        return w, h

    def _cam_vel_from_H(self, frame: np.ndarray) -> np.ndarray:
        """
        Camera velocity (dx, dy) in pixels from the last computed homography.
        Computed as displacement of the frame centre under H.
        Returns zeros if H is unavailable.
        """
        if self._last_H is None:
            return np.zeros(2)
        h_fr, w_fr = frame.shape[:2]
        cx, cy = w_fr / 2.0, h_fr / 2.0
        denom  = (self._last_H[2,0]*cx + self._last_H[2,1]*cy
                  + self._last_H[2,2] + 1e-8)
        ncx    = ((self._last_H[0,0]*cx + self._last_H[0,1]*cy
                   + self._last_H[0,2]) / denom)
        ncy    = ((self._last_H[1,0]*cx + self._last_H[1,1]*cy
                   + self._last_H[1,2]) / denom)
        return np.array([ncx - cx, ncy - cy])

    def _effective_dist_sigma(self, frame: np.ndarray) -> float:
        """
        Compute dist_sigma for DRM spatial penalty.

        Scales with BOTH object size AND EKF positional uncertainty so that
        the Gaussian distance penalty weakens automatically when the EKF is
        highly uncertain (i.e. large ROI / long occlusion).  This prevents
        candidates near a stale EKF prediction from being unfairly favoured
        over the correct object which may have drifted further away.
        """
        obj_w, obj_h = self._get_median_size()
        size_sigma   = self._drm_dist_sigma_factor * float(max(obj_w, obj_h))
        ekf_sigma    = self.ekf.get_uncertainty() * 1.5
        return max(size_sigma, ekf_sigma)

    def _build_candidate_velocities(
    self,
    last_idx: int,
) -> List[Optional[np.ndarray]]:
        """
        For each candidate in _cand_frames[last_idx], require it to have a
        matched detection in EVERY prior collection frame (0 … last_idx-1).
        If the candidate is missing from ANY frame → return None for it
        (caller will exclude it from DRM entirely).

        Velocity is computed as the weighted average of per-frame displacements
        across the full track, camera-compensated by the accumulated homography
        data from each step.
        """
        last_frame = self._cand_frames[last_idx]
        if not last_frame:
            return []

        # If this is the only collection frame there are no prior frames to
        # match against — every candidate is considered "fully tracked" but
        # velocity is unknown (caller treats None as neutral, not excluded).
        # We only EXCLUDE when there ARE prior frames and the candidate misses one.
        n_prior = last_idx   # number of frames before last_idx (indices 0…last_idx-1)

        results: List[Optional[np.ndarray]] = []

        for (bbox_last, desc_last) in last_frame:
            cx_last = float(bbox_last[0] + bbox_last[2] / 2.0)
            cy_last = float(bbox_last[1] + bbox_last[3] / 2.0)

            if n_prior == 0:
                # Single collection frame: no velocity computable, but don't exclude
                results.append(None)
                continue

            # ── Require match in every prior frame ────────────────────────────
            per_frame_cx = []   # cx at each frame index 0 … last_idx
            per_frame_cy = []
            all_found = True

            for j in range(last_idx):   # must find in ALL frames 0 … last_idx-1
                early_frame = self._cand_frames[j]
                if not early_frame:
                    all_found = False
                    break

                best_score  = 0.35   # minimum match quality threshold
                best_cx_e   = None
                best_cy_e   = None

                for (bbox_e, desc_e) in early_frame:
                    match = (0.55 * _iou(bbox_last, bbox_e)
                            + 0.45 * _cos_sim(desc_last, desc_e))
                    if match > best_score:
                        best_score = match
                        best_cx_e  = float(bbox_e[0] + bbox_e[2] / 2.0)
                        best_cy_e  = float(bbox_e[1] + bbox_e[3] / 2.0)

                if best_cx_e is None:
                    # No acceptable match in frame j → candidate is not fully tracked
                    all_found = False
                    break

                per_frame_cx.append(best_cx_e)
                per_frame_cy.append(best_cy_e)

            if not all_found:
                results.append(None)   # will be excluded from DRM
                continue

            # Append the last-frame position to complete the track
            per_frame_cx.append(cx_last)
            per_frame_cy.append(cy_last)

            # ── Compute per-step displacements and camera-compensate each step ─
            # _occ_cam_vels[k] = camera vel AT collection frame k
            #                    (motion from frame k-1 → frame k, 1-indexed)
            step_vels = []
            for step in range(last_idx):   # steps: 0→1, 1→2, …, (last_idx-1)→last_idx
                raw_dx = per_frame_cx[step + 1] - per_frame_cx[step]
                raw_dy = per_frame_cy[step + 1] - per_frame_cy[step]

                # cam vel from frame[step] → frame[step+1] is _occ_cam_vels[step+1]
                # (index step+1 because index 0 = cam vel at first collection frame,
                #  which itself is motion from the frame BEFORE collection started)
                cam_idx = step + 1
                if cam_idx < len(self._occ_cam_vels):
                    cam = self._occ_cam_vels[cam_idx]
                else:
                    cam = np.zeros(2)

                step_vels.append(np.array([raw_dx - cam[0], raw_dy - cam[1]],
                                        dtype=float))

            # Mean camera-compensated velocity over the full track
            found_vel = np.mean(step_vels, axis=0) if step_vels else np.zeros(2)
            results.append(found_vel)

        return results

    def _compute_velocity_score(self, cand_vel, expected_vel) -> float:
        expected_speed = float(np.linalg.norm(expected_vel))
        cand_speed     = float(np.linalg.norm(cand_vel))
        if expected_speed < self._vel_score_min_speed:
            return 0.5   # neutral — stationary/noise EKF vel, don't bias

        cos = float(np.dot(expected_vel, cand_vel) /
                    (expected_speed * (cand_speed + 1e-8)))
        cos = float(np.clip(cos, -1.0, 1.0))
        dir_score = (cos + 1.0) / 2.0   # ∈ [0, 1]

        # Speed score is GATED by direction:
        # If direction is wrong (cos < 0) speed agreement is irrelevant /
        # suspicious — clamp speed_score contribution to 0 in that case.
        speed_ratio = cand_speed / (expected_speed + 1e-8)
        speed_score = float(np.clip(1.0 - abs(speed_ratio - 1.0), 0.0, 1.0))
        if cos < 0.0:
            speed_score = 0.0   # speed match on a wrong-direction cand is not a reward

        # Direction weight raised to 0.80 — direction matters more than speed magnitude
        raw = 0.80 * dir_score + 0.20 * speed_score   # ∈ [0, 1]

        # Hard gate: strongly opposite direction (cos < -threshold) → floor at 0.05
        # This ensures a near-180° wrong candidate can never score neutrally.
        if cos < -self._vel_dir_hard_gate:
            raw = min(raw, 0.05)

        return float(np.clip(raw, 0.0, 1.0))

    # ─────────────────────────────────────────────────────────────────────────
    # Background homography estimation  (UNCHANGED)
    # ─────────────────────────────────────────────────────────────────────────

    def _estimate_homography(
        self, frame: np.ndarray
    ) -> Tuple[Optional[np.ndarray], bool, np.ndarray]:
        _LK_PARAMS = dict(
            winSize  = (20, 20),
            maxLevel = 2,
            criteria = (
                cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 8, 0.04,
            ),
        )
        SCALE = 0.5

        small = cv2.resize(frame, None, fx=SCALE, fy=SCALE,
                           interpolation=cv2.INTER_LINEAR)
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        if self.prev_gray is None:
            return None, False, gray

        if self.prev_gray.shape != gray.shape:
            prev_gray_scaled = cv2.resize(
                self.prev_gray, (gray.shape[1], gray.shape[0]),
                interpolation=cv2.INTER_LINEAR)
        else:
            prev_gray_scaled = self.prev_gray

        H        = None
        reliable = False

        if self._cached_shape != gray.shape:
            step = 50
            yg, xg = np.mgrid[
                step // 2 : gray.shape[0] : step,
                step // 2 : gray.shape[1] : step,
            ]
            self._cached_pts   = np.column_stack(
                (xg.ravel(), yg.ravel())).astype(np.float32)
            self._cached_shape = gray.shape

        grid = self._cached_pts

        ref_box = self.held_box if self.held_box is not None else self.current_bbox
        if ref_box is not None:
            x, y, w, h = (v * SCALE for v in map(int, ref_box))
            pad = max(4, int(max(w, h) * 0.15))
            inside = (
                (grid[:, 0] >= x - pad) & (grid[:, 0] < x + w + pad) &
                (grid[:, 1] >= y - pad) & (grid[:, 1] < y + h + pad)
            )
            pts = grid[~inside].reshape(-1, 1, 2)
        else:
            pts = grid.reshape(-1, 1, 2)

        if len(pts) < 6:
            return H, reliable, gray

        new_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray_scaled, gray, pts, None, **_LK_PARAMS)

        if status is None:
            return H, reliable, gray

        ok       = status.ravel() == 1
        good_old = pts[ok]
        good_new = new_pts[ok]

        if len(good_old) < 6:
            return H, reliable, gray

        A, inliers = cv2.estimateAffinePartial2D(
            good_old, good_new,
            method                = cv2.RANSAC,
            ransacReprojThreshold = 3.0,
            maxIters              = 500,
            confidence            = 0.99,
        )

        if A is not None and inliers is not None:
            inlier_ratio = float(inliers.sum()) / len(good_old)
            reliable     = inlier_ratio >= self.homo_inlier_threshold
            H            = np.eye(3, dtype=np.float64)
            H[:2, :]     = A
            H[0, 2]     /= SCALE
            H[1, 2]     /= SCALE

        return H, reliable, gray

    # ── Nudge toward nearest detection  (UNCHANGED) ───────────────────────────

    def _nudge_toward_nearest(self, frame: np.ndarray,
                               detections: List[np.ndarray]) -> np.ndarray:
        ref  = self.memory.best_descriptor()
        hcx  = self.held_box[0] + self.held_box[2] / 2.0
        hcy  = self.held_box[1] + self.held_box[3] / 2.0

        best_det, best_rank = None, float('inf')
        for det in detections:
            dcx  = det[0] + det[2] / 2.0
            dcy  = det[1] + det[3] / 2.0
            dist = np.hypot(dcx - hcx, dcy - hcy)
            if ref is not None:
                desc = _extract_descriptor(frame, det)
                sim  = _cos_sim(ref, desc) if desc is not None else 0.0
                rank = dist * (1.0 - 0.5 * sim)
            else:
                rank = dist
            if rank < best_rank:
                best_rank, best_det = rank, det

        if best_det is None:
            return self.held_box.copy()

        dcx    = best_det[0] + best_det[2] / 2.0
        dcy    = best_det[1] + best_det[3] / 2.0
        new_cx = hcx + self.nudge_alpha * (dcx - hcx)
        new_cy = hcy + self.nudge_alpha * (dcy - hcy)

        hw, hh     = self.held_box[2], self.held_box[3]
        h_fr, w_fr = frame.shape[:2]
        nx = int(np.clip(new_cx - hw / 2.0, 0, w_fr - 1))
        ny = int(np.clip(new_cy - hh / 2.0, 0, h_fr - 1))
        return np.array([nx, ny,
                         int(np.clip(hw, 1, w_fr - nx)),
                         int(np.clip(hh, 1, h_fr - ny))], dtype=int)

    # ── YOLO search ROI  (CHANGED: tiny-object ROI parameter set) ────────────

    def _get_yolo_search_roi(self, frame: np.ndarray) -> Tuple[int, int, int, int]:
        h_fr, w_fr = frame.shape[:2]
        max_side   = min(w_fr, h_fr)

        if self.frame_idx <= 30 or self.recovered_early_occlusion == False:
            self.recovered_early_occlusion = False
            scale = 0.4
            bw    = int(w_fr * scale)
            bh    = int(h_fr * scale)
            x     = (w_fr - bw) // 2
            y     = (h_fr - bh) // 2
            return x, y, bw, bh

        # CHANGED: choose ROI parameter set based on object scale.
        # When the object is tiny / long-distance, a wider but more tightly
        # expanding ROI is used so we don't immediately inflate to the full frame.
        # For out-of-frame we always use the normal set (edge-scan behaviour).
        is_tiny = (not self._out_of_frame) and self._is_long_distance(frame)
        if is_tiny:
            _roi_start_expand            = self.tiny_roi_start_expand
            _yolo_search_expand          = self.tiny_yolo_search_expand
            _search_expand_growth_factor = self.tiny_search_expand_growth_factor
            _search_expand_growth_every  = self.tiny_search_expand_growth_every
        else:
            _roi_start_expand            = self.roi_start_expand
            _yolo_search_expand          = self.yolo_search_expand
            _search_expand_growth_factor = self.search_expand_growth_factor
            _search_expand_growth_every  = self.search_expand_growth_every

        obj_w, obj_h = self._get_median_size()
        obj_size     = (obj_w + obj_h) // 2

        steps            = self._occ_frames // max(1, _search_expand_growth_every)
        time_expand      = float(_search_expand_growth_factor ** steps)
        effective_expand = min(_roi_start_expand * time_expand, _yolo_search_expand)

        if self._out_of_frame and self._exit_edge is not None:
            side = max(1, int(obj_size * effective_expand))

            if self._exit_edge == 'right':
                if (self._search_cx - w_fr) > obj_w * 2:
                    return 0, 0, 0, 0
                scy    = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
                x1, y1 = w_fr - side, int(scy - side // 2)
                rw, rh  = side, side

            elif self._exit_edge == 'left':
                scy    = float(np.clip(self._search_cy, side // 2, h_fr - side // 2))
                x1, y1 = 0, int(scy - side // 2)
                rw, rh  = side, side

            elif self._exit_edge == 'bottom':
                scx    = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
                x1, y1 = int(scx - side // 2), h_fr - side
                rw, rh  = side, side

            else:   # top
                scx    = float(np.clip(self._search_cx, side // 2, w_fr - side // 2))
                x1, y1 = int(scx - side // 2), 0
                rw, rh  = side, side

        else:
            half   = min((obj_size * effective_expand) / 2.0, max_side / 2.0)
            cx_lo  = min(half, w_fr / 2.0)
            cy_lo  = min(half, h_fr / 2.0)
            scx    = float(np.clip(self._search_cx, cx_lo, w_fr - cx_lo))
            scy    = float(np.clip(self._search_cy, cy_lo, h_fr - cy_lo))
            side   = max(1, int(half * 2))
            x1, y1 = int(scx - half), int(scy - half)
            rw, rh  = side, side

        # Final clamp
        x1 = max(0, x1)
        y1 = max(0, y1)
        rw = min(rw, w_fr - x1)
        rh = min(rh, h_fr - y1)
        rw = max(1, rw)
        rh = max(1, rh)
        return x1, y1, rw, rh

    # ── YOLO (with per-frame cache)  (UNCHANGED) ──────────────────────────────

    def _yolo_detect(self, frame: np.ndarray) -> List[np.ndarray]:
        rx, ry, rw, rh = self._get_yolo_search_roi(frame)
        crop           = frame[ry:ry + rh, rx:rx + rw]
        if crop.size == 0:
            self._yolo_cache = []
            return []
        results = self.yolo.predict(crop, conf=self.yolo_conf,
                                    iou=self.yolo_iou_thr, verbose=False,
                                    imgsz=320)
        boxes = []
        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                xyxy   = box.xyxy[0].cpu().numpy()
                cls_id = int(box.cls[0].cpu().numpy())
                # Filter by class if locked
                if (self._yolo_filter_class
                        and self._target_class_id is not None
                        and cls_id != self._target_class_id):
                    continue
                x1, y1, x2, y2 = xyxy
                boxes.append(np.array([int(x1)+rx, int(y1)+ry,
                                        int(x2-x1), int(y2-y1)], dtype=int))
        self._yolo_cache = boxes
        return boxes
    
    def _yolo_detect_cached(self, frame: np.ndarray) -> List[np.ndarray]:
        if self._yolo_cache:
            return self._yolo_cache
        return self._yolo_detect(frame)

    # ── Misc helpers  (UNCHANGED) ─────────────────────────────────────────────

    def _clamp_bbox_to_frame(self, bbox: np.ndarray, frame: np.ndarray) -> np.ndarray:
        h_fr, w_fr = frame.shape[:2]
        x, y, w, h = bbox
        w = int(np.clip(w, 1, w_fr))
        h = int(np.clip(h, 1, h_fr))
        x = int(np.clip(x, -w, w_fr))
        y = int(np.clip(y, -h, h_fr))
        return np.array([x, y, w, h], dtype=int)

    def _h_translation_magnitude(self, H, frame: np.ndarray) -> float:
        if H is None:
            return 0.0
        h, w   = frame.shape[:2]
        cx, cy = w / 2.0, h / 2.0
        denom  = H[2, 0]*cx + H[2, 1]*cy + H[2, 2] + 1e-8
        new_cx = (H[0, 0]*cx + H[0, 1]*cy + H[0, 2]) / denom
        new_cy = (H[1, 0]*cx + H[1, 1]*cy + H[1, 2]) / denom
        return float(np.hypot(new_cx - cx, new_cy - cy))

    def _classify_loss_cause(
        self,
        cam_disp_threshold:    float = 18.0,
        area_shrink_threshold: float = -0.005,
    ) -> str:
        history  = list(self._conf_history)
        cam_hist = list(self._cam_disp_history)

        area_vote = 'occlusion'
        if len(history) >= 3:
            areas      = np.array([float(b[2] * b[3]) for b, _, _, _ in history])
            med_area   = float(np.median(areas))
            n          = len(areas)
            t          = np.arange(n, dtype=float)
            slope      = float(np.polyfit(t, areas, 1)[0])
            norm_slope = slope / (med_area + 1e-6)
            if norm_slope >= area_shrink_threshold:
                area_vote = 'camera_motion'

        cam_vote = 'occlusion'
        if cam_hist:
            weights  = np.exp(np.linspace(-1, 0, len(cam_hist)))
            weights /= weights.sum()
            mean_disp = float(np.dot(weights, cam_hist))
            if mean_disp >= cam_disp_threshold:
                cam_vote = 'camera_motion'

        if area_vote == 'camera_motion' and cam_vote == 'camera_motion':
            return 'camera_motion'
        return 'occlusion'

    def _is_long_distance(self, frame: np.ndarray) -> bool:
        if self.long_distance_mode:
            return True
        if not self._size_history:
            return False
        h_fr, w_fr  = frame.shape[:2]
        frame_area  = float(h_fr * w_fr)
        obj_w       = float(np.median([s[0] for s in self._size_history]))
        obj_h       = float(np.median([s[1] for s in self._size_history]))
        obj_area    = obj_w * obj_h
        return (obj_area / (frame_area + 1e-8)) < self.long_distance_area_fraction

    def _rebuild_ekf_from_clean_history(self, skip_override=None) -> BBoxEKF:
        history  = list(self._conf_history)
        raw_skip = self.history_skip_last if skip_override is None else skip_override
        skip     = min(raw_skip, max(0, len(history) - 2))
        clean    = history[:len(history) - skip] if skip > 0 else history

        if len(clean) == 0:
            return self.ekf

        first_bbox, _, _, _ = clean[0]
        fresh_ekf = BBoxEKF(
            first_bbox,
            process_noise = self.ekf_process_noise,
            meas_noise    = self.ekf_meas_noise,
        )

        for (bbox, _vel, h, h_rel) in clean[1:]:
            fresh_ekf.predict(H=h, H_reliable=(h is not None))
            fresh_ekf.update(bbox)

        robust_vel = self._robust_velocity_from_history(
            skip=skip, window=self.velocity_window_average,
            decay=0.97, clip_percentile=96.0)
        fresh_ekf.x[2]     = float(robust_vel[0])
        fresh_ekf.x[3]     = float(robust_vel[1])
        fresh_ekf.P[2, 2]  = 40.0
        fresh_ekf.P[3, 3]  = 40.0

        return fresh_ekf

    def _detect_shrinkage_onset(
        self,
        max_lookback:  int   = 10,
        min_drop_frac: float = 0.002,
        smooth_k:      int   = 3,
    ) -> int:
        history = list(self._conf_history)
        n = len(history)
        if n < 4:
            return 0

        areas    = np.array([float(b[2] * b[3]) for b, _, _, _ in history],
                             dtype=float)
        smoothed = np.array([
            float(np.median(areas[max(0, i - smooth_k + 1): i + 1]))
            for i in range(n)
        ])

        ref_area  = float(np.percentile(smoothed, 95))
        if ref_area <= 0:
            return 0

        threshold  = ref_area * (1.0 - min_drop_frac)
        skip       = 0
        gap_budget = 3
        gaps_used  = 0

        for i in range(n - 1, max(n - 1 - max_lookback, -1), -1):
            if smoothed[i] < threshold:
                skip      += 1
                gaps_used  = 0
            elif skip > 0 and gaps_used < gap_budget:
                skip      += 1
                gaps_used += 1
            else:
                break

        if skip >= max_lookback:
            full_drop = (ref_area - float(np.mean(smoothed[n - skip:]))) / (ref_area + 1e-6)
            if full_drop < min_drop_frac:
                return 0

        return max(skip, 5)

    def _detect_center_drift_skip(
        self,
        max_lookback: int   = 20,
        spike_factor: float = 2.5,
    ) -> int:
        hist = list(self._center_history)
        if len(hist) < 4:
            return 0

        centers = np.stack(hist)
        speeds  = np.linalg.norm(np.diff(centers, axis=0), axis=1)
        n       = len(speeds)

        ref_n   = max(2, n * 2 // 3)
        ref_mag = float(np.median(speeds[:ref_n])) + 1e-6
        thresh  = ref_mag * spike_factor

        skip       = 0
        gap_budget = 2
        gaps_used  = 0

        for i in range(n - 1, max(n - 1 - max_lookback, -1), -1):
            if speeds[i] > thresh:
                skip      += 1
                gaps_used  = 0
            elif skip > 0 and gaps_used < gap_budget:
                skip      += 1
                gaps_used += 1
            else:
                break

        return skip

    def _detect_exit_direction(
        self,
        frame:            np.ndarray,
        lookahead_frames: int   = 6,
        trend_frames:     int   = 10,
        margin_factor:    float = 0.5,
    ) -> Tuple[bool, Optional[str]]:
        h_fr, w_fr = frame.shape[:2]

        last_bbox        = self.current_bbox.astype(float)
        lx1, ly1, lw, lh = last_bbox
        lx2, ly2         = lx1 + lw, ly1 + lh
        lcx              = lx1 + lw / 2.0
        lcy              = ly1 + lh / 2.0

        vx = float(self.velocity[0])
        vy = float(self.velocity[1])

        margin      = max(lw, lh) * margin_factor
        prox_right  = lx2 > w_fr - margin and vx > 0
        prox_left   = lx1 < margin         and vx < 0
        prox_bottom = ly2 > h_fr - margin  and vy > 0
        prox_top    = ly1 < margin         and vy < 0

        fut_cx         = lcx + vx * lookahead_frames
        fut_cy         = lcy + vy * lookahead_frames
        extrap_right   = fut_cx >= w_fr
        extrap_left    = fut_cx < 0
        extrap_bottom  = fut_cy >= h_fr
        extrap_top     = fut_cy < 0

        history  = list(self._conf_history)
        n_use    = min(trend_frames, len(history))
        trend_right = trend_left = trend_bottom = trend_top = False

        if n_use >= 3:
            recent   = history[-n_use:]
            xs = np.array([float(b[0] + b[2] / 2) for b, *_ in recent])
            ys = np.array([float(b[1] + b[3] / 2) for b, *_ in recent])
            t  = np.arange(n_use, dtype=float)

            vx_trend = float(np.polyfit(t, xs, 1)[0])
            vy_trend = float(np.polyfit(t, ys, 1)[0])

            fut_tx       = lcx + vx_trend * lookahead_frames
            fut_ty       = lcy + vy_trend * lookahead_frames
            trend_right  = fut_tx >= w_fr  and vx_trend > 0
            trend_left   = fut_tx < 0      and vx_trend < 0
            trend_bottom = fut_ty >= h_fr  and vy_trend > 0
            trend_top    = fut_ty < 0      and vy_trend < 0

        def _votes(right, left, bottom, top):
            return {'right': right, 'left': left, 'bottom': bottom, 'top': top}

        e1 = _votes(prox_right,   prox_left,   prox_bottom,   prox_top)
        e2 = _votes(extrap_right, extrap_left, extrap_bottom, extrap_top)
        e3 = _votes(trend_right,  trend_left,  trend_bottom,  trend_top)

        for edge in ('right', 'left', 'bottom', 'top'):
            if sum([e1[edge], e2[edge], e3[edge]]) >= 2:
                return True, edge

        if lx2 >= w_fr: return True, 'right'
        if lx1 <= 0:    return True, 'left'
        if ly2 >= h_fr: return True, 'bottom'
        if ly1 <= 0:    return True, 'top'

        return False, None

    def _robust_velocity_from_history(
        self, skip=0, window=80, decay=0.97, clip_percentile=80.0
    ) -> np.ndarray:
        hist     = list(self._center_history)
        if len(hist) < 2:
            return self.velocity.copy()

        conf_len = len(list(self._conf_history))
        hist     = hist[-conf_len:] if len(hist) > conf_len else hist

        if skip > 0:
            hist = hist[:max(2, len(hist) - skip)]

        hist = hist[-window:] if len(hist) > window else hist
        if len(hist) < 2:
            return self.velocity.copy()

        centers    = np.stack(hist)
        frame_vels = np.diff(centers, axis=0)
        n          = len(frame_vels)

        weights    = np.array([decay ** (n - 1 - i) for i in range(n)], dtype=float)
        weights   /= weights.sum()

        avg = (weights[:, None] * frame_vels).sum(axis=0)

        cam_hist = list(self._cam_vel_history)
        cam_hist = cam_hist[-conf_len:] if len(cam_hist) > conf_len else cam_hist
        if skip > 0:
            cam_hist = cam_hist[:max(2, len(cam_hist) - skip)]
        cam_hist = cam_hist[-window:] if len(cam_hist) > window else cam_hist

        if len(cam_hist) >= n:
            cam_vels = np.stack(cam_hist[-n:])
            avg_cam  = (weights[:, None] * cam_vels).sum(axis=0)
            avg     -= avg_cam

        magnitudes = np.linalg.norm(frame_vels, axis=1)
        mag_cap    = float(np.percentile(magnitudes, clip_percentile))
        avg_mag    = float(np.linalg.norm(avg))
        if avg_mag > mag_cap and avg_mag > 0:
            avg = avg * (mag_cap / avg_mag)

        return avg
    

    # ── New helper: class warm-up ─────────────────────────────────────────────
    def _try_detect_target_class(self, frame: np.ndarray) -> None:
        """
        During the first `yolo_class_detect_frames` frames run YOLO on the
        current bbox area and vote for which class overlaps most with it.
        Sets self._target_class_id once confident.
        Called only when yolo_filter_class=True and _class_warmup_done=False.
        """
        if self.current_bbox is None:
            return
        x, y, w, h = self.current_bbox
        # Small pad so we don't miss slightly misaligned detections
        pad = max(10, int(max(w, h) * 0.2))
        h_fr, w_fr = frame.shape[:2]
        x1 = max(0, x - pad);  y1 = max(0, y - pad)
        x2 = min(w_fr, x + w + pad);  y2 = min(h_fr, y + h + pad)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            return

        results = self.yolo.predict(crop, conf=self.yolo_conf,
                                    iou=self.yolo_iou_thr, verbose=False,
                                    imgsz=320)
        if not results or results[0].boxes is None:
            return

        best_iou, best_cls = 0.0, None
        for box in results[0].boxes:
            bx1, by1, bx2, by2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu().numpy())
            det = np.array([int(bx1)+x1, int(by1)+y1,
                            int(bx2-bx1), int(by2-by1)], dtype=int)
            iou = _iou(det, self.current_bbox)
            if iou > best_iou:
                best_iou, best_cls = iou, cls_id

        if best_iou >= 0.3 and best_cls is not None:
            if not hasattr(self, '_class_votes'):
                self._class_votes: dict = {}
            self._class_votes[best_cls] = self._class_votes.get(best_cls, 0) + 1

    def _maybe_commit_target_class(self) -> None:
        votes = getattr(self, '_class_votes', {})
        if not votes:
            return
        best_cls = max(votes, key=votes.get)
        total    = sum(votes.values())
        if votes[best_cls] / total >= 0.6:   # 60% agreement → commit
            self._target_class_id  = best_cls
            self._class_warmup_done = True
            print(f"[class filter] target class locked: {best_cls}  "
                f"(votes={votes})")
        else:
            # No consensus yet — stay unlocked, keep voting
            pass

    @property
    def running_dynamic_bbox(self):
        return self.tracker.running_dynamic_bbox

    @property
    def running_dynamic_image(self):
        return self.tracker.running_dynamic_image

    @property
    def tracking_config(self):
        return self.tracker.tracking_config

    @property
    def tracking_state(self):
        return self.tracker.tracking_state


# ─────────────────────────────────────────────────────────────────────────────
# Example initialisation showing all new parameters
# ─────────────────────────────────────────────────────────────────────────────
#
# tracker = EdgeDAMTracker(
#     siam_tracker        = wrapped,
#     yolo_weights        = "yolo11n.pt",
#
#     # ── Core thresholds ───────────────────────────────────────────────────
#     conf_threshold      = 0.5,      # score BELOW this starts the entry streak
#     reacq_threshold     = 0.7,      # tracker score to exit occlusion (phase 0)
#     yolo_conf           = 0.3,
#     app_match_threshold = 0.6,      # cosine sim to accept tracker bbox as target
#     nudge_alpha         = 0.0,
#
#     # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
#     entry_patience      = 3,        # N consecutive bad frames before occlusion
#
#     # ── NEW: Multi-frame candidate collection ─────────────────────────────
#     cand_collection_frames = 3,     # YOLO collection frames before final DRM
#
#     # ── NEW: Velocity scoring weight & guard ──────────────────────────────
#     drm_lam_cand_vel    = 0.20,     # weight of vel_score in final DRM phase
#                                     # set 0 to disable velocity scoring
#     vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
#                                     # if EKF speed < this → vel_score = 0.5 (neutral)
#
#     # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
#     # Used automatically when _is_long_distance() returns True.
#     tiny_roi_start_expand            = 3.0,
#     tiny_yolo_search_expand          = 20.0,
#     tiny_search_expand_growth_factor = 1.3,
#     tiny_search_expand_growth_every  = 5,
#     tiny_search_expand_max           = 40.0,
#
#     # ── DRM (existing params) ─────────────────────────────────────────────
#     drm_tau_sim         = 0.70,
#     mem_capacity        = 50,
#     drm_mmin            = 5,
#     drm_capacity        = 20,
#     history_decay       = 0.1,
#     drm_lam_dist        = 0.0,
#     drm_lam_cand_dir    = 0.0,
#     drm_lam_time        = 0.0,
#     drm_lam_app         = 0.9,
#     drm_lam_iou         = 0.0,
#     drm_lam_mot         = 0.1,
#     drm_margin          = 0.7,
#     drm_skip_threshold  = 23,
#     drm_top_k           = 100,
#
#     # ── History ───────────────────────────────────────────────────────────
#     conf_history_len    = 200,
#     size_history_len    = 200,
#     history_skip_last   = 8,
#
#     # ── ROI expansion (normal objects) ────────────────────────────────────
#     roi_start_expand                 = 20,
#     yolo_search_expand               = 100,
#     search_expand_growth_factor      = 1.4,
#     search_expand_growth_every       = 75,
#     search_expand_max                = 500.0,
#
#     # ── Misc ──────────────────────────────────────────────────────────────
#     velocity_window_average          = 50,
#     shrinkage_max_lookback           = 30,
#     enter_occlusion_on_loss          = True,
#     drm_gamma                        = 0,
# )

In [9]:
from collections import defaultdict
import gc
from logging import config
import torch
from collections import deque

test_public_lb = manifest["public_lb"]
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json" , "r") as f:
    manifest = json.load(f)
manifest.keys()
test_public_lb = manifest["public_lb"]

video_paths = [
    "/home/moha/AIC-4/data_competition/dataset5/uav2/uav2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/uav7/uav7_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/RcCar6/RcCar6_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/Motor1/Motor1_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/MountainBike5/MountainBike5_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset1/person_3/person_3.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/truck/truck_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/human3/human3_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset1/sheeps_2/sheeps_2.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/Animal3/Animal3_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/person16/person16_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/electric_box/electric_box_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/RcCar4/RcCar4_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset4/group2/group2_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/jogging2/jogging2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/car6_2/car6_2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset4/person19/person19_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/couple/couple_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/tennis_player1_2/tennis_player1_2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/car4/car4_96.mp4"
]




# tracker = EdgeDAMTracker(
#     siam_tracker         = wrapped,
#     yolo_weights         = "yolo11n.pt",
#     conf_threshold       = 0.5,   # below this → enter occlusion
#     reacq_threshold      = 0.7,   # tracker score to exit occlusion
#     yolo_conf            = 0.3,
#     app_match_threshold  = 0.6,   # cosine sim to accept a YOLO det as the target
#     nudge_alpha          = 0.0,   # how far to step toward nearest det per frame
#     drm_tau_sim=0.70,
#     mem_capacity = 50,
#     drm_mmin=5,
#     drm_capacity = 20,
#     history_decay = 0.1,
#     drm_lam_dist = 0.0,
#     drm_lam_cand_dir=0.0,
#     # drm_lam_cand_vel = 0.2,
#     drm_lam_time=0.0,
#     # drm_lam_cand_vel=0.,
#     drm_lam_app=0.9,
#     drm_lam_iou  = 0 ,
#     drm_lam_mot=0.1,
#     drm_margin = 0.7,
#     drm_skip_threshold = 23,
#     drm_top_k=100,
#     conf_history_len = 200,
#     size_history_len = 200,
#     history_skip_last=8,
#     roi_start_expand = 20,
#     yolo_search_expand = 100,
#     search_expand_growth_factor= 1.4,   # ← multiply ROI by this each N frames
#     search_expand_growth_every  = 75,     # ← N frames
#     search_expand_max = 500.0,  # ← cap so it doesn't eat the whole frame
#     velocity_window_average = 50,
#     shrinkage_max_lookback= 30,
#     enter_occlusion_on_loss = True,
#     drm_gamma = 0,
    
    
#     # ekf_meas_noise = 50,

#     # app_spatial_weight = 0.0
#     # occ_patience = 30 ,
#     )
from external.SiamABC.realtime_test import load_hydra_config_from_path , get_tracker 

model_size = "M"
weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_new/head_epoch_000.pth"
# weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_2/head_epoch_001.pth"
config_path = "../external/SiamABC/core/config"
config_name = "SiamABC_tracker"

config = load_hydra_config_from_path(config_path=config_path, config_name=config_name)
config["model"]["model_size"] = 'S' if model_size=="S_Tiny" else 'M'
config["tracker"]["N"] =10
config["tracker"]["lr"] = 0.4
config["tracker"]["dynamic_update"] = True 
config["tracker"]["memory_window_size"] = 25
config["tracker"]["dynamic_update_threshold"] = 0.7
config["tracker"]["running_confidence_floor_value"] = 100
config["tracker"]["search_context"] = 2
config["tracker"]["iou_threshold"] = 0.6


# config["tracker"]["window_influence"] =  0.45 
# config["tracker"]["penalty_k"] = 0.10


wrapped = get_tracker(config=config, weights_path=weights_path , lambda_tta=0.1 , continuous=False)
# siam.all_memory_imgs = deque(maxlen=250)   # was 5000
# siam.classification_scores = deque(maxlen=250)


tracker = EdgeDAMTracker(
    siam_tracker        = wrapped,
    yolo_weights        = "yolo11n.pt",

    # ── Core thresholds ───────────────────────────────────────────────────
    conf_threshold      = 0.4,      # score BELOW this starts the entry streak
    reacq_threshold     = 0.70,      # tracker score to exit occlusion (phase 0)
    yolo_conf           = 0.3,
    app_match_threshold = 0.65,      # cosine sim to accept tracker bbox as target
    nudge_alpha         = 0.0,

    # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
    entry_patience      = 1,        # N consecutive bad frames before occlusion

    # ── NEW: Multi-frame candidate collection ─────────────────────────────
    cand_collection_frames = 1,     # YOLO collection frames before final DRM

    # ── NEW: Velocity scoring weight & guard ──────────────────────────────
    drm_lam_cand_vel    = 0.0,     # weight of vel_score in final DRM phase
                                    # set 0 to disable velocity scoring
    vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
                                    # if EKF speed < this → vel_score = 0.5 (neutral)

    # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
    # Used automatically when _is_long_distance() returns True.
    long_distance_area_fraction =  0.030,
    tiny_roi_start_expand            = 8.0,
    tiny_yolo_search_expand          = 20.0,
    tiny_search_expand_growth_factor = 1.3,
    tiny_search_expand_growth_every  = 50,
    tiny_search_expand_max           = 40.0,

    # ── DRM (existing params) ─────────────────────────────────────────────
    drm_tau_sim         = 0.60,
    mem_capacity        = 50,
    drm_mmin            = 4,
    drm_capacity        = 20,
    history_decay       = 0.1,
    drm_lam_dist        = 0.0,
    drm_lam_cand_dir    = 0.0,
    drm_lam_time        = 0.0,
    drm_lam_app         = 1,
    drm_lam_iou         = 0.0,
    drm_lam_mot         = 0.0,
    drm_margin          = 0.45,
    drm_skip_threshold  = 23,
    drm_top_k           = 100,
    vel_dir_hard_gate = 0.4,   # |cos| threshold below which score → 0.05
    yolo_filter_class  = False, # filter candidates to target class
    yolo_class_detect_frames   = 5,     # stride 

    # ── History ───────────────────────────────────────────────────────────
    conf_history_len    = 200,
    size_history_len    = 200,
    history_skip_last   = 8,

    # ── ROI expansion (normal objects) ────────────────────────────────────
    roi_start_expand                 = 20,
    yolo_search_expand               = 100,
    search_expand_growth_factor      = 1.4,
    search_expand_growth_every       = 150,
    search_expand_max                = 500.0,

    # ── Misc ──────────────────────────────────────────────────────────────
    velocity_window_average          = 200,
    shrinkage_max_lookback           = 30,
    enter_occlusion_on_loss          = True,
    drm_gamma                        = 0,


)
start = 1
for i in range(start , start+len(video_paths)):
    video_path = video_paths[i-start]
    ann_path = os.path.join(os.path.dirname(video_path) , "annotation.txt")
    output_path = f"outputs/test_5_after/test_{i}.mp4"



    init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()
    if not isinstance(init_bbox[0] , (int , float)):
        init_bbox =init_bbox[0]
    print(init_bbox)

    #     # ── Force-clear tracker memory BEFORE loading new video ──────────────
    # if hasattr(tracker, 'all_memory_imgs'):
    #     tracker.all_memory_imgs.clear()
    # if hasattr(tracker, 'classification_scores'):
    #     tracker.classification_scores.clear()
    # if hasattr(tracker, '_template_features') and tracker._template_features is not None:
    #     del tracker._template_features
    #     tracker._template_features = None
    # if hasattr(tracker, 'dynamic_template_features'):
    #     del tracker.dynamic_template_features
    # if hasattr(tracker, 'dynamic_search_features'):
    #     del tracker.dynamic_search_features

    # torch.cuda.empty_cache()
    # gc.collect()

    run_inference(
        video_path=video_path,
        initial_bbox=init_bbox,
        tracker=tracker,
        output_path=output_path
    )
        


{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 25, 'dynamic_update_threshold': 0.7, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6}
[344.0, 236.0, 8.0, 6.0]
[occlusion entry] frame=30  loss_cause=occlusion  out_of_frame=False  exit_edge=None  entry_streak=1
[occlusion entry] frame=30  loss_cause=occlusion  dynamic_skip=17  entry_streak_skip=1  effective_skip=17  history_len=29
[occ frame 2] phase=collect(1/1)  detections=1  stored=1
[occ frame 3] phase=final_drm  last_cands=1  fully_tracked=1  drm_size=2  ram=5  ekf_unc=20.7px
[occ frame 5] phase=collect(1/1)  detections=1  stored=1
[occ frame 6] phase=final_drm  last_cands=1  fully_tracked=1  drm_size=2  ram=5  ekf_unc=44.3px
[occ

In [ ]:
test_public_lb

{'dataset1/Car_video': {'dataset': 'dataset1',
  'seq_name': 'Car_video',
  'n_frames': 585,
  'native_fps': 60,
  'video_path': 'dataset1/Car_video/Car_video.mp4',
  'annotation_path': 'dataset1/Car_video/annotation.txt'},
 'dataset1/Car_video_4': {'dataset': 'dataset1',
  'seq_name': 'Car_video_4',
  'n_frames': 415,
  'native_fps': 30,
  'video_path': 'dataset1/Car_video_4/Car_video_4.mp4',
  'annotation_path': 'dataset1/Car_video_4/annotation.txt'},
 'dataset1/basketball': {'dataset': 'dataset1',
  'seq_name': 'basketball',
  'n_frames': 324,
  'native_fps': 60,
  'video_path': 'dataset1/basketball/basketball.mp4',
  'annotation_path': 'dataset1/basketball/annotation.txt'},
 'dataset1/basketball_2': {'dataset': 'dataset1',
  'seq_name': 'basketball_2',
  'n_frames': 184,
  'native_fps': 30,
  'video_path': 'dataset1/basketball_2/basketball_2.mp4',
  'annotation_path': 'dataset1/basketball_2/annotation.txt'},
 'dataset1/basketball_3': {'dataset': 'dataset1',
  'seq_name': 'basketbal

In [10]:

#### dam_4
import pandas as pd 

test_public_lb = manifest["public_lb"]
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json" , "r") as f:
    manifest = json.load(f)
manifest.keys()
test_public_lb = manifest["public_lb"]

# test_pub_2 = {k:v for k , v in manifest["public_lb"].items() if v["dataset"]=="dataset2"}


folder_names = [
    # "uav2",
    # "uav7",
    # "RcCar6",
    "Motor1",
    "MountainBike5",
    "person_3",
    "truck",
    "human3",
    "sheeps_2",
    "Animal3",
    "person16",
    "electric_box",
    "RcCar4",
    "group2",
    "truck",
    "jogging2",
    "car6_2",
    "person19",
    "couple",
    "tennis_player1_2",
    "car4",
]


data_dir = "../data_competition"
outputs_dir = "../outputs/SiamABC_updated_dam"
test_public_lb = {
    k:v for k , v in manifest["public_lb"].items() 
    if v["dataset"] in [ 
         "dataset1" , 
           "dataset2", 
           "dataset3" , 
           "dataset4" , 
           "dataset5" 
           ]
    # and v["seq_name"] in  folder_names
    }



for i , (key , value) in enumerate(test_public_lb.items()):
    
    
    model_size = "M"
    weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_new/head_epoch_000.pth"
    # weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_2/head_epoch_001.pth"
    config_path = "../external/SiamABC/core/config"
    config_name = "SiamABC_tracker"

    config = load_hydra_config_from_path(config_path=config_path, config_name=config_name)
    config["model"]["model_size"] = 'S' if model_size=="S_Tiny" else 'M'
    config["tracker"]["N"] =10
    config["tracker"]["lr"] = 0.4
    config["tracker"]["dynamic_update"] = True 
    config["tracker"]["memory_window_size"] = 25
    config["tracker"]["dynamic_update_threshold"] = 0.7
    config["tracker"]["running_confidence_floor_value"] = 100
    config["tracker"]["search_context"] = 2
    config["tracker"]["iou_threshold"] = 0.6


    # config["tracker"]["window_influence"] =  0.45 
    # config["tracker"]["penalty_k"] = 0.10


    wrapped = get_tracker(config=config, weights_path=weights_path , lambda_tta=0.1 , continuous=False)
    # siam.all_memory_imgs = deque(maxlen=250)   # was 5000
    # siam.classification_scores = deque(maxlen=250)


    tracker = EdgeDAMTracker(
        siam_tracker        = wrapped,
        yolo_weights        = "yolo11n.pt",

        # ── Core thresholds ───────────────────────────────────────────────────
        conf_threshold      = 0.4,      # score BELOW this starts the entry streak
        reacq_threshold     = 0.70,      # tracker score to exit occlusion (phase 0)
        yolo_conf           = 0.3,
        app_match_threshold = 0.65,      # cosine sim to accept tracker bbox as target
        nudge_alpha         = 0.0,

        # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
        entry_patience      = 1,        # N consecutive bad frames before occlusion

        # ── NEW: Multi-frame candidate collection ─────────────────────────────
        cand_collection_frames = 1,     # YOLO collection frames before final DRM

        # ── NEW: Velocity scoring weight & guard ──────────────────────────────
        drm_lam_cand_vel    = 0.0,     # weight of vel_score in final DRM phase
                                        # set 0 to disable velocity scoring
        vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
                                        # if EKF speed < this → vel_score = 0.5 (neutral)

        # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
        # Used automatically when _is_long_distance() returns True.
        long_distance_area_fraction =  0.030,
        tiny_roi_start_expand            = 8.0,
        tiny_yolo_search_expand          = 20.0,
        tiny_search_expand_growth_factor = 1.3,
        tiny_search_expand_growth_every  = 50,
        tiny_search_expand_max           = 40.0,

        # ── DRM (existing params) ─────────────────────────────────────────────
        drm_tau_sim         = 0.60,
        mem_capacity        = 50,
        drm_mmin            = 4,
        drm_capacity        = 20,
        history_decay       = 0.1,
        drm_lam_dist        = 0.0,
        drm_lam_cand_dir    = 0.0,
        drm_lam_time        = 0.0,
        drm_lam_app         = 1,
        drm_lam_iou         = 0.0,
        drm_lam_mot         = 0.0,
        drm_margin          = 0.45,
        drm_skip_threshold  = 23,
        drm_top_k           = 100,
        vel_dir_hard_gate = 0.4,   # |cos| threshold below which score → 0.05
        yolo_filter_class  = False, # filter candidates to target class
        yolo_class_detect_frames   = 5,     # stride 

        # ── History ───────────────────────────────────────────────────────────
        conf_history_len    = 200,
        size_history_len    = 200,
        history_skip_last   = 8,

        # ── ROI expansion (normal objects) ────────────────────────────────────
        roi_start_expand                 = 20,
        yolo_search_expand               = 100,
        search_expand_growth_factor      = 1.4,
        search_expand_growth_every       = 150,
        search_expand_max                = 500.0,

        # ── Misc ──────────────────────────────────────────────────────────────
        velocity_window_average          = 200,
        shrinkage_max_lookback           = 30,
        enter_occlusion_on_loss          = True,
        drm_gamma                        = 0,


    )


    print(f"Processing video {i+1}/{len(test_public_lb)}: {value['video_path']}")
    video_path = os.path.join(
        data_dir , value["video_path"]
    )
    ann_path = os.path.join(
        data_dir , value["annotation_path"]
    )

    output_path = os.path.join(
        outputs_dir , value["video_path"]
    )


    init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()


    run_inference(
        video_path=video_path,
        initial_bbox=init_bbox,
        tracker=tracker,
        output_path=output_path
    )
    
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json" , "r") as f:
    test_public_lb = json.load(f)["public_lb"]
submission_df = defaultdict(list)
for key, value in test_public_lb.items():
    video_path = os.path.join(data_dir, value["video_path"])
    output_path = os.path.join(outputs_dir, value["video_path"])
    print(video_path , output_path)
    
    head, tail = os.path.split(output_path)
    bbox_dir = os.path.join(head, 'bboxes')
    bbox_file = os.path.join(bbox_dir, os.path.splitext(tail)[0] + '.txt')
    
    seq_id = os.path.splitext(value["video_path"])[0]
    if os.path.exists(bbox_file):
        with open(bbox_file, 'r') as f:
            lines = f.read().strip().split('\n')
            
            for frame_idx, line in enumerate(lines):
                x, y, w, h = line.strip().split()
                submission_df["id"].append(f"{key}_{frame_idx}")
                submission_df["x"].append(float(x))
                submission_df["y"].append(float(y))
                submission_df["w"].append(float(w))
                submission_df["h"].append(float(h))

submission = pd.DataFrame(submission_df)
submission.to_csv("OrTrack.csv" , index = False)

print(submission.head())
print(f"Total rows: {len(submission)}")

{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 25, 'dynamic_update_threshold': 0.7, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6}
Processing video 1/89: dataset1/Car_video/Car_video.mp4
ALL FRAMES            n= 584  mean=20.6ms  med=17.7ms  p95=40.8ms  p99=52.1ms  min=12.1ms  max=64.1ms  fps=48.6

─── Latency Report ───────────────────────────────────────
NORMAL TRACK          n= 584  mean=20.6ms  med=17.7ms  p95=40.8ms  p99=52.1ms  min=12.1ms  max=64.1ms  fps=48.6

─── Latency Report ───────────────────────────────────────
OCCLUSION             no data
──────────────────────────────────────────────────────────

{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing

# system description 
    - 4 pages 
    - novelty 

# github repository
    - readme (how to run the repo)
    - training scripts 
    - evaluation
    - documented (human-like)
    - clean (add .yaml files for model config)
    - docker file
    - github actions
    - example outputs
    - model weights to drive
    - compile model with TRT. 
    - script to covert video data to png data
    - github page

    -directories:
        
        -utils/
            -utility functions


        -train/
            -finetuning scripts
        
        -test/
            -evaluate.py --> submission.csv   (python evaluate.py)
            -inference.py  --> output.mp4

        -models/ -> but they usually have different models, small, medium, large
            -SiamRAM.py  (Recovery )

        -checkpoints/  --> generated folder
            -model_checkpoint.pth

        -data/ ---> generated folder
        

        readme.md -> includes everything to run repo end to end 

        






#






In [12]:
submission.to_csv("SiamABC_latest.csv" , index = False)

print(submission.head())
print(f"Total rows: {len(submission)}")

                     id      x      y      w      h
0  dataset1/Car_video_0  536.0  551.0  226.0  142.0
1  dataset1/Car_video_1  533.0  549.0  237.0  150.0
2  dataset1/Car_video_2  531.0  549.0  237.0  152.0
3  dataset1/Car_video_3  531.0  549.0  237.0  152.0
4  dataset1/Car_video_4  529.0  550.0  239.0  155.0
Total rows: 74293
